# 台指期（TX）快速修復回檔入場策略 — FinLab 回測（Colab 版）

大跌之後如果**修復得夠快**，行情大概率會回到舊高。本 notebook 把這個觀察寫成規則，
訊號來自**加權指數**，部位開在**台指期連續合約**，用 `finlab.backtest.sim()` 回測
並產生 `report.display()` 的互動報表。

程式碼與換倉價差都已嵌入，**不需 clone repo**，結果與 repo 的
`scripts/futures_trades.py`／`dist/tx_futures_backtest.py` 完全一致。

**執行方式**：由上而下依序執行即可。需要 FinLab API token。

---
### 與 ETF 版（00631L）的四個差異

1. **換倉**：近月到期時，以「換倉日的近月收盤」平倉、「次月同日收盤」建倉，
   價差不計入損益。台指期長期逆價差，不還原的話 300 多次換倉會憑空造成
   每年約 −3.8% 的假虧損（實測未還原 7.28x vs 已還原 20.44x）。
2. **槓桿由停損距離決定**：L = 風險預算 ÷ 停損距離，上限 `MAX_LEVERAGE`。
   刻意不套 ETF 版的 5% 停損下限 —— 那個下限會讓 L 恆等於 1.6 倍，上限永遠碰不到。
3. **進場後不調整口數**：持有期間權益為 1 + L×(F/F0 − 1)，線性、不複利、不再平衡。
4. **當天成交**：加權指數 13:30 收盤、台指期 13:45 收盤，中間 15 分鐘足夠下單。
   訊號以指數收盤判定後，當天的期貨收盤就能成交，不必等隔天（ETF 版同時收盤，只能次日）。

> ⚠️ 這是研究專案，**不是投資建議**。即使當天成交，5 倍上限仍有 −45.7% 的
> 最大回檔與單筆 −22% 的虧損，原因見末尾的「停損假設 vs 實際」一節。


## 1. 安裝套件


In [ ]:
!pip install -q finlab


## 2. 登入 FinLab

執行後貼上你的 API token（不會顯示在畫面上）。
token 可在 [FinLab 會員頁面](https://ai.finlab.tw/) 取得。


In [ ]:
import getpass, os, warnings
import finlab

token = (os.environ.get('FINLAB_API_TOKEN') or os.environ.get('Finlab_API_token')
         or getpass.getpass('FinLab API token: '))
with warnings.catch_warnings():
    warnings.simplefilter('ignore', DeprecationWarning)
    finlab.login(token)


## 3. 寫入策略程式碼

以下是 `tw_backdraw` 套件的完整原始碼，直接寫成檔案後 import ——
與專案回測使用的是同一份程式，不是簡化版。


In [ ]:
import pathlib

PKG = pathlib.Path('tw_backdraw')
PKG.mkdir(exist_ok=True)
SOURCES = {}

SOURCES['bars'] = '"""日線資料結構與讀檔。"""\n\nfrom __future__ import annotations\n\nimport csv\nfrom dataclasses import dataclass\nfrom datetime import date, datetime\nfrom pathlib import Path\n\n\n@dataclass(frozen=True)\nclass Bar:\n    d: date\n    open: float\n    high: float\n    low: float\n    close: float\n\n    @property\n    def iso(self) -> str:\n        return self.d.isoformat()\n\n\ndef _parse_date(raw: str) -> date:\n    raw = raw.strip()\n    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%Y%m%d"):\n        try:\n            return datetime.strptime(raw, fmt).date()\n        except ValueError:\n            continue\n    raise ValueError(f"無法解析日期: {raw!r}")\n\n\ndef load_csv(path: str | Path) -> list[Bar]:\n    """讀取日線 CSV。\n\n    必要欄位: date, close。open/high/low 缺漏時以 close 補齊，\n    這樣只有收盤價的資料集也能直接跑（策略訊號全部以收盤價判定）。\n    """\n    bars: list[Bar] = []\n    with open(path, newline="", encoding="utf-8-sig") as fh:\n        for row in csv.DictReader(fh):\n            row = {(k or "").strip().lower(): (v or "").strip() for k, v in row.items()}\n            if not row.get("date") or not row.get("close"):\n                continue\n            close = float(row["close"].replace(",", ""))\n\n            def pick(key: str) -> float:\n                val = row.get(key, "")\n                return float(val.replace(",", "")) if val else close\n\n            bars.append(\n                Bar(\n                    d=_parse_date(row["date"]),\n                    open=pick("open"),\n                    high=pick("high"),\n                    low=pick("low"),\n                    close=close,\n                )\n            )\n    bars.sort(key=lambda b: b.d)\n    if not bars:\n        raise ValueError(f"{path} 沒有可用的日線資料")\n    return bars\n'
SOURCES['config'] = '"""策略參數。\n\n所有可調參數集中於此，方便做敏感度測試。\n預設值對應貼文中的歷史統計（12 國、近百年、174 次樣本）。\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\n\n@dataclass(frozen=True)\nclass SetupConfig:\n    """「快速修復」劇本的辨識條件。"""\n\n    # 先要有一段像樣的回檔才談得上修復，貼文樣本為 -16%\n    min_drawdown: float = 0.10\n    # 從谷底起算，補回跌幅的多少比例才算「補回來了」。\n    # 貼文的原始值是 0.75；預設改用 grid search 以「總報酬 ÷ 最大回檔」選出的 0.60。\n    repair_fraction: float = 0.60\n    # 補回必須在幾個交易日內完成。\n    # 貼文的原始值是 15（對應「89% 創高」那一組）；預設改用 0.60/30 這組。\n    max_repair_bars: int = 30\n    # 訊號有效期；超過仍未創高也未失效就自然過期\n    setup_expiry_bars: int = 120\n    # 修復視窗過期後，把參考高點重新錨定到「谷底之後的波段高」。\n    # 關掉的話，參考高點會一直釘在舊高，直到指數重新站上它為止 ——\n    # 台股 2000 年頭部之後花了 17.3 年才收復 10,202，等於中間完全看不到訊號。\n    reanchor_on_expiry: bool = True\n\n\n@dataclass(frozen=True)\nclass LevelConfig:\n    """由 P（前高）與 T（谷底）推出的關鍵價位。"""\n\n    # 主防線：補回一半\n    half_line_ratio: float = 0.50\n    # 警戒線：貼文的原始值是 0.382（台股 42916）。\n    # 預設改用 0.500 —— 這會讓警戒線與主防線重合，\n    # 等於「跌破補回一半的線（含 0.5% 緩衝）就全部出場」，是明顯更緊的停損。\n    warn_line_ratio: float = 0.500\n    # 允許對主防線的假跌破緩衝（貼文：一半案例跌破不到 0.5%）\n    half_line_buffer: float = 0.005\n\n\n@dataclass(frozen=True)\nclass EntryConfig:\n    """分批進場梯。\n\n    核心前提：歷史上回檔中位數只有 2.9%、四分之三不超過 5%，\n    所以「等深回檔」的期望值是負的 —— 底倉先上車，回檔才是加碼機會。\n    """\n\n    # 訊號確認後隔日直接建立的底倉權重。\n    # 貼文原意是 0.40（保留兩段回檔加碼空間）；預設改用 1.00，\n    # 也就是訊號一確認就把目標水位一次買足，不留加碼梯。\n    base_weight: float = 1.00\n    # (自訊號後波段高點的回檔幅度, 加碼權重)\n    pullback_ladder: tuple[tuple[float, float], ...] = ((0.03, 0.30), (0.05, 0.30))\n    # 幾個交易日內若都沒等到回檔，就以市價補齊剩餘部位（不參與才是最大風險）。\n    # 註：預設 base_weight=1.00 時沒有加碼梯，本參數不影響任何決策；\n    # grid search 對它的取值也近乎均勻分布（10/20/40 各約三分之一），維持 20 為中性值。\n    fill_timeout_bars: int = 20\n    # 突破前高後補齊剩餘部位。\n    # 預設 True：訊號觸發時距前高通常只剩 3~4%，等不到回檔梯就先創高是常態，\n    # 若此時取消加碼，實際部署會長期停在底倉水位（實測平均僅 24%）。\n    # 往上買的單位風險較高，引擎會按停損距離自動縮小這一段的權重。\n    breakout_fills_remainder: bool = True\n\n\n@dataclass(frozen=True)\nclass ExitConfig:\n    """出場與風控。"""\n\n    # 收盤跌破警戒線 → 減碼比例。\n    # 貼文原意是 0.50（砍一半、保留船票）；預設改用 1.00，也就是跌破就全部出場。\n    # 這會降低勝率、提高總報酬與報酬÷回檔（見 docs/strategy.md §11）。\n    warn_derisk_fraction: float = 1.00\n    # 收盤跌破谷底 → 全出（貼文中 11% 的失敗案例都是大熊市開場）\n    hard_stop_at_trough: bool = True\n    # 觸及前高後先落袋的比例。\n    # 預設 0：「離前高很近，本身不是賣出的理由」——前高只是統計上的高機率目標，\n    # 不是出場訊號；獲利全部交給停利機制處理。設 1/3 可改回分批落袋。\n    target_take_fraction: float = 0.0\n\n    # 創高之後用哪一種停利：\n    #   "trail"       自創高後最高收盤回檔 trail_drawdown → 出場\n    #   "ma_ratchet"  停利價 = max(前高, MA)。創高後先把前高當出場線，\n    #                 等 MA 爬過前高，就改看 MA 跌破 —— 均線只會把出場線往上推。\n    #   "both"        兩者取較緊（較高）的那一條\n    exit_mode: str = "trail"\n    # ma_ratchet / both 使用的均線天期（以加權指數收盤計）\n    ma_period: int = 20\n    # trail / both 使用的回檔幅度（以指數計）\n    trail_drawdown: float = 0.08\n\n\n@dataclass(frozen=True)\nclass SizingConfig:\n    """部位規模（標的為 2 倍槓桿 ETF，必須以指數停損距離反推）。"""\n\n    # 單筆交易願意承受的權益風險（兩段式停損全走完的預期損失）\n    risk_per_trade: float = 0.08\n    # 標的槓桿倍數（台灣50正2 = 2）\n    leverage: float = 2.0\n    # 最高持股水位（佔權益比例）\n    max_weight: float = 1.0\n    # 停損距離至少視為這麼大，避免訊號日離谷底太近而算出過大部位\n    min_stop_distance: float = 0.05\n\n\n@dataclass(frozen=True)\nclass CostConfig:\n    """台股 ETF 交易成本與槓桿 ETF 的內扣損耗。"""\n\n    # 券商手續費 0.1425% × 折扣\n    fee_rate: float = 0.001425\n    fee_discount: float = 0.60\n    # ETF 賣出證交稅 0.1%\n    tax_rate: float = 0.001\n    # 槓桿 ETF 年化內扣（管理費 + 期貨轉倉/避險成本）\n    annual_carry: float = 0.012\n    trading_days: int = 252\n\n    @property\n    def buy_cost(self) -> float:\n        return self.fee_rate * self.fee_discount\n\n    @property\n    def sell_cost(self) -> float:\n        return self.fee_rate * self.fee_discount + self.tax_rate\n\n    @property\n    def daily_carry(self) -> float:\n        return self.annual_carry / self.trading_days\n\n\n@dataclass(frozen=True)\nclass StrategyConfig:\n    setup: SetupConfig = field(default_factory=SetupConfig)\n    levels: LevelConfig = field(default_factory=LevelConfig)\n    entry: EntryConfig = field(default_factory=EntryConfig)\n    exit: ExitConfig = field(default_factory=ExitConfig)\n    sizing: SizingConfig = field(default_factory=SizingConfig)\n    cost: CostConfig = field(default_factory=CostConfig)\n\n\n# ---------------------------------------------------------------------------\n# 預設組（PRESETS）\n#\n# 以下各組是 648,000 組 grid search + 樣本外驗證（scripts/grid_search.py、\n# scripts/walk_forward.py）的產物。**預設是 tuned**，也就是以總報酬為目標\n# 選出來的那一組；忠於貼文的原始設定保留在 `post`。\n# 兩者的取捨與已知風險記在 docs/strategy.md §11，換組合前請先讀。\n# ---------------------------------------------------------------------------\n\n#: 以「總報酬 ÷ 最大回檔」為目標選出的參數（本專案的預設）。\n#: 放寬訊號定義（30 日 / 補回 60%）、訊號確認即滿倉、跌破主防線全數出場。\n#: 樣本內 21 筆、勝率 57%、總報酬 +2232%、最大回檔 -35.8%、比值 62.4。\n#: 樣本外驗證中，這個目標函數的表現優於直接用總報酬（見 docs/strategy.md §11）。\nTUNED_CONFIG = StrategyConfig()\n\nDEFAULT_CONFIG = TUNED_CONFIG\n\n#: 忠於貼文的原始設定：15 日內補回 75%、底倉四成留加碼梯、\n#: 38.2% 回補位減碼一半。樣本內 8 筆、勝率 50%、總報酬 +31.4%、最大回檔 -20.4%。\nPOST_CONFIG = StrategyConfig(\n    setup=SetupConfig(repair_fraction=0.75, max_repair_bars=15),\n    levels=LevelConfig(warn_line_ratio=0.382),\n    entry=EntryConfig(base_weight=0.40),\n    exit=ExitConfig(warn_derisk_fraction=0.50),\n)\n\n#: 調校過的訊號 + MA40 棘輪出場。用報酬換勝率與較小的回檔：\n#: 樣本內 21 筆、勝率 67%、總報酬 +459%、回檔 -35%。\nBALANCED_CONFIG = StrategyConfig(\n    exit=ExitConfig(warn_derisk_fraction=1.0, exit_mode="ma_ratchet", ma_period=40),\n)\n\n#: 全網格勝率最高的一組。**它是靠關掉兩道停損換來的**，總報酬遠低於預設組。\n#: 列在這裡是為了讓「最大化勝率」的後果可以被重現，不是建議值。\nWINRATE_CONFIG = StrategyConfig(\n    setup=SetupConfig(min_drawdown=0.13),\n    levels=LevelConfig(warn_line_ratio=0.382),\n    entry=EntryConfig(fill_timeout_bars=40),\n    exit=ExitConfig(warn_derisk_fraction=0.0, hard_stop_at_trough=False,\n                    exit_mode="ma_ratchet", ma_period=40),\n)\n\n#: 美股版（S&P 500 訊號 → UPRO 3x 執行）以「總報酬 ÷ 最大回檔」選出的參數。\n#: 與台股版的差異：回檔門檻 7%（S&P 500 在 FinLab 涵蓋的 10.6 年裡 ≥10% 的\n#: 回檔太少）、移動停利放寬到 10%（3 倍槓桿的波動較大）。\n#: 樣本內 8 筆、勝率 62%、總報酬 +627%、最大回檔 -37.5%、比值 16.7。\n#: ⚠️ min_drawdown 落在搜尋網格的下界，最佳值可能在網格之外 —— 見 docs/strategy.md §12。\nUS_TUNED_CONFIG = StrategyConfig(\n    setup=SetupConfig(min_drawdown=0.07, repair_fraction=0.60, max_repair_bars=30),\n    levels=LevelConfig(warn_line_ratio=0.500),\n    entry=EntryConfig(base_weight=1.0, fill_timeout_bars=10),\n    exit=ExitConfig(warn_derisk_fraction=1.0, exit_mode="trail", trail_drawdown=0.10),\n    sizing=SizingConfig(leverage=3.0),\n    cost=CostConfig(fee_rate=0.0, fee_discount=1.0, tax_rate=0.0, annual_carry=0.0091),\n)\n\n#: NASDAQ 版（^IXIC 訊號 → TQQQ 3x 執行）以「總報酬 ÷ 最大回檔」選出的參數。\n#: 樣本內 8 筆、勝率 88%、總報酬 +933%、最大回檔 -30.8%、比值 30.3。\n#:\n#: ⚠️ trail_drawdown=0.12 是**尖峰而非平台**：鄰近值的比值為 10%→19.0、\n#: 12%→30.3、15%→15.7，且交易數從 12 筆掉到 8 筆。這是過擬合的典型特徵，\n#: 較穩健的鄰居是 trail=0.08（16 筆、比值 21.3）。詳見 docs/strategy.md §13。\nNQ_TUNED_CONFIG = StrategyConfig(\n    setup=SetupConfig(min_drawdown=0.07, repair_fraction=0.60, max_repair_bars=30),\n    levels=LevelConfig(warn_line_ratio=0.382),\n    entry=EntryConfig(base_weight=1.0, fill_timeout_bars=10),\n    exit=ExitConfig(warn_derisk_fraction=1.0, exit_mode="trail", trail_drawdown=0.12),\n    sizing=SizingConfig(leverage=3.0),\n    cost=CostConfig(fee_rate=0.0, fee_discount=1.0, tax_rate=0.0, annual_carry=0.0095),\n)\n\n#: 同上但把移動停利改成鄰域穩健的 8%：16 筆、勝率 62%、+570%、-26.8%、比值 21.3。\nNQ_ROBUST_CONFIG = StrategyConfig(\n    setup=NQ_TUNED_CONFIG.setup, levels=NQ_TUNED_CONFIG.levels,\n    entry=NQ_TUNED_CONFIG.entry,\n    exit=ExitConfig(warn_derisk_fraction=1.0, exit_mode="trail", trail_drawdown=0.08),\n    sizing=NQ_TUNED_CONFIG.sizing, cost=NQ_TUNED_CONFIG.cost,\n)\n\nPRESETS: dict[str, StrategyConfig] = {\n    "tuned": TUNED_CONFIG,\n    "post": POST_CONFIG,\n    "balanced": BALANCED_CONFIG,\n    "winrate": WINRATE_CONFIG,\n    "us_tuned": US_TUNED_CONFIG,\n    "nq_tuned": NQ_TUNED_CONFIG,\n    "nq_robust": NQ_ROBUST_CONFIG,\n}\n'
SOURCES['levels'] = '"""由「前高 P」與「谷底 T」推導出的關鍵價位。\n\n台股當前實例：P = 47742、T = 39933、跌幅 R = 7809 點\n    主防線 (50%)  = 43837   ← 貼文的 43800\n    警戒線 (38.2%) = 42916\n    失效線        = 39933\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom .config import LevelConfig\n\n\n@dataclass(frozen=True)\nclass Levels:\n    peak: float\n    trough: float\n    half_line: float\n    half_line_with_buffer: float\n    warn_line: float\n    invalidation: float\n\n    @property\n    def drop(self) -> float:\n        return self.peak - self.trough\n\n    @property\n    def drop_pct(self) -> float:\n        return self.drop / self.peak\n\n    @property\n    def stop_line(self) -> float:\n        """真正會觸發出場的價位。\n\n        `zone()` 要落到 warning 得同時跌破警戒線**與**主防線的假跌破緩衝區，\n        所以實際的停損線是兩者取低。預設參數下警戒線與主防線重合\n        （`warn_line_ratio == half_line_ratio`），緩衝區就成了真正的觸發點 ——\n        收盤跌破主防線 0.5% 以內會被容忍，超過才出場。\n        """\n        return min(self.warn_line, self.half_line_with_buffer)\n\n    def repair_fraction(self, price: float) -> float:\n        """價格相當於補回跌幅的幾成。"""\n        if self.drop <= 0:\n            return 0.0\n        return (price - self.trough) / self.drop\n\n    def zone(self, price: float) -> str:\n        """把價格歸類到操作分區。"""\n        if price >= self.peak:\n            return "breakout"       # 已創高：轉移動停利\n        if price >= self.half_line:\n            return "healthy"        # 劇本正常：回檔即加碼\n        if price >= self.half_line_with_buffer:\n            return "buffer"         # 假跌破容忍區：不加碼、不減碼\n        if price >= self.warn_line:\n            return "caution"        # 跌破主防線：停止加碼\n        if price >= self.invalidation:\n            return "warning"        # 跌破 38.2%：減碼\n        return "invalidated"        # 跌破谷底：另一個故事，全出\n\n\ndef build_levels(peak: float, trough: float, cfg: LevelConfig) -> Levels:\n    if peak <= trough:\n        raise ValueError("前高必須高於谷底")\n    drop = peak - trough\n    half = trough + cfg.half_line_ratio * drop\n    return Levels(\n        peak=peak,\n        trough=trough,\n        half_line=half,\n        half_line_with_buffer=half * (1 - cfg.half_line_buffer),\n        warn_line=trough + cfg.warn_line_ratio * drop,\n        invalidation=trough,\n    )\n'
SOURCES['setup'] = '"""辨識「快速修復」劇本。\n\n流程（全部以收盤價、可即時判定，不使用未來資料）：\n    1. 從滾動高點 P 起算，收盤跌破 P×(1-10%) → 進入回檔追蹤\n    2. 追蹤期間持續更新谷底 T（創更低就更新，計時歸零）\n    3. 自 T 起 N 個交易日內，收盤補回跌幅 ≥ 75% → 觸發訊號\n       （N ≤ 15 對應貼文中「89% 會先回到舊高點」的那一組）\n    4. 超過 15 日才補回 → 這一段作廢（那一組成功率只剩 48%，跟丟銅板一樣），\n       參考高點改錨到谷底之後的波段高，重新開始找下一組 P/T\n\n第 4 步的改錨很關鍵。少了它，參考高點會一直釘在舊高直到指數重新站上為止 ——\n台股 2000 年頭部之後花了 17.3 年才收復 10,202，中間包含 2008、2015、2020\n在內的所有回檔修復都會被那個舊高遮蔽，偵測器等於瞎掉。\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom datetime import date\n\nfrom .bars import Bar\nfrom .config import SetupConfig\n\n\n@dataclass(frozen=True)\nclass Episode:\n    """一段 ≥10% 的回檔，以及它為什麼（沒）觸發訊號。"""\n\n    peak: float\n    peak_date: date\n    peak_index: int\n    trough: float\n    trough_date: date\n    trough_index: int\n    lower_lows: int              # 追蹤期間破底幾次（每次都讓計時歸零）\n    best_in_window: float        # 15 日視窗內補回的最高比例\n    best_in_window_bars: int     # 上述最佳值出現在谷底後第幾日\n    fired: bool\n    end_index: int\n    end_date: date\n\n    @property\n    def drop_pct(self) -> float:\n        return (self.trough - self.peak) / self.peak\n\n    def reason(self, cfg: SetupConfig) -> str:\n        if self.fired:\n            return f"✅ {self.best_in_window_bars} 日補回 {self.best_in_window:.0%}"\n        return (f"❌ {cfg.max_repair_bars} 日內只補回 {self.best_in_window:.0%}"\n                f"（第 {self.best_in_window_bars} 日最佳）")\n\n\n@dataclass(frozen=True)\nclass FastRepairSetup:\n    peak: float\n    peak_date: date\n    trough: float\n    trough_date: date\n    trough_index: int\n    trigger_index: int\n    trigger_date: date\n    trigger_close: float\n    bars_to_repair: int\n    repair_fraction: float\n\n    @property\n    def drop_pct(self) -> float:\n        return (self.peak - self.trough) / self.peak\n\n    def describe(self) -> str:\n        return (\n            f"{self.trough_date} 谷底 {self.trough:,.0f}（自 {self.peak_date} 高點 "\n            f"{self.peak:,.0f} 回檔 {self.drop_pct:.1%}），"\n            f"{self.bars_to_repair} 個交易日補回 {self.repair_fraction:.0%}，"\n            f"{self.trigger_date} 觸發訊號 @ {self.trigger_close:,.0f}"\n        )\n\n\n@dataclass(frozen=True)\nclass LiveState:\n    """偵測器跑到最後一根 K 的當下狀態 —— 每日訊號報告用。\n\n    直接從 `scan_episodes()` 的同一個迴圈取得，不另外實作一份，\n    所以盤中看到的「還差多少觸發」與回測的判定必然一致。\n    """\n\n    state: str                   # normal / drawdown / expired / engaged\n    peak: float\n    peak_date: date\n    trough: float | None         # 只有 drawdown 狀態才有\n    trough_date: date | None\n    bars_since_trough: int | None\n    repair_fraction: float | None    # 目前補回幾成\n    best_in_window: float | None     # 視窗內補回過的最高比例\n    lower_lows: int\n    bars_left: int | None            # 修復視窗還剩幾個交易日\n\n    @property\n    def drop_pct(self) -> float | None:\n        if self.trough is None or self.peak <= 0:\n            return None\n        return (self.peak - self.trough) / self.peak\n\n    def trigger_close(self, cfg: SetupConfig) -> float | None:\n        """再收在多少以上就會觸發訊號。"""\n        if self.state != "drawdown" or self.trough is None:\n            return None\n        return self.trough + cfg.repair_fraction * (self.peak - self.trough)\n\n\ndef scan_episodes(bars: list[Bar], cfg: SetupConfig,\n                  live: list | None = None) -> list[Episode]:\n    """列出所有 ≥ `min_drawdown` 的回檔段落，含觸發與未觸發的原因。\n\n    `detect_setups()` 就是取這裡面 `fired=True` 的那些，所以兩者不會不一致。\n\n    傳入 `live=[]` 時，會把跑到最後一根 K 的 `LiveState` 追加進去 ——\n    每日訊號報告靠這個取得「目前追蹤到哪、還差多少觸發」，\n    而不是另外複製一份狀態機。\n    """\n    episodes: list[Episode] = []\n    if not bars:\n        return episodes\n\n    peak, peak_i = bars[0].close, 0\n    trough, trough_i = bars[0].close, 0\n    state = "normal"\n    lower_lows = 0\n    win_best, win_best_n = 0.0, 0\n\n    def flush(i: int, fired: bool) -> None:\n        episodes.append(Episode(\n            peak=peak, peak_date=bars[peak_i].d, peak_index=peak_i,\n            trough=trough, trough_date=bars[trough_i].d, trough_index=trough_i,\n            lower_lows=lower_lows, best_in_window=win_best,\n            best_in_window_bars=win_best_n, fired=fired,\n            end_index=i, end_date=bars[i].d,\n        ))\n\n    for i, bar in enumerate(bars):\n        c = bar.close\n\n        if state == "normal":\n            if c > peak:\n                peak, peak_i = c, i\n            elif c <= peak * (1 - cfg.min_drawdown):\n                state = "drawdown"\n                trough, trough_i = c, i\n                lower_lows, win_best, win_best_n = 0, 0.0, 0\n            continue\n\n        if state == "drawdown":\n            if c < trough:\n                # 創更低點：谷底、計時與視窗內最佳進度一起重設\n                trough, trough_i = c, i\n                lower_lows += 1\n                win_best, win_best_n = 0.0, 0\n                continue\n\n            elapsed = i - trough_i\n            frac = (c - trough) / (peak - trough) if peak > trough else 0.0\n            if elapsed <= cfg.max_repair_bars and frac > win_best:\n                win_best, win_best_n = frac, elapsed\n\n            if frac >= cfg.repair_fraction and elapsed <= cfg.max_repair_bars:\n                flush(i, fired=True)\n                state = "engaged"\n                continue\n\n            if elapsed > cfg.max_repair_bars:\n                # 修復視窗到期，這一段作廢\n                flush(i, fired=False)\n                if cfg.reanchor_on_expiry:\n                    seg = bars[trough_i:i + 1]\n                    k = max(range(len(seg)), key=lambda j: seg[j].close)\n                    peak, peak_i = seg[k].close, trough_i + k\n                    state = "normal"\n                    continue\n                # 不改錨：等指數重新站上舊高才解除\n                state = "expired"\n                continue\n\n            if c > peak:\n                flush(i, fired=False)\n                state, peak, peak_i = "normal", c, i\n            continue\n\n        if state == "expired":\n            if c > peak:\n                state, peak, peak_i = "normal", c, i\n            continue\n\n        if state == "engaged":\n            last = episodes[-1]\n            if (c > peak or c < last.trough\n                    or (i - last.end_index) > cfg.setup_expiry_bars):\n                state = "normal"\n                if c > peak:\n                    peak, peak_i = c, i\n            continue\n\n    if live is not None and bars:\n        i = len(bars) - 1\n        if state == "drawdown":\n            elapsed = i - trough_i\n            drop = peak - trough\n            live.append(LiveState(\n                state=state, peak=peak, peak_date=bars[peak_i].d,\n                trough=trough, trough_date=bars[trough_i].d,\n                bars_since_trough=elapsed,\n                repair_fraction=(bars[i].close - trough) / drop if drop > 0 else 0.0,\n                best_in_window=win_best, lower_lows=lower_lows,\n                bars_left=max(cfg.max_repair_bars - elapsed, 0)))\n        else:\n            live.append(LiveState(\n                state=state, peak=peak, peak_date=bars[peak_i].d,\n                trough=None, trough_date=None, bars_since_trough=None,\n                repair_fraction=None, best_in_window=None,\n                lower_lows=0, bars_left=None))\n\n    return episodes\n\n\ndef detect_setups(bars: list[Bar], cfg: SetupConfig) -> list[FastRepairSetup]:\n    """掃描整段歷史，回傳所有觸發過的快速修復訊號。"""\n    return [\n        FastRepairSetup(\n            peak=e.peak, peak_date=e.peak_date,\n            trough=e.trough, trough_date=e.trough_date, trough_index=e.trough_index,\n            trigger_index=e.end_index, trigger_date=e.end_date,\n            trigger_close=bars[e.end_index].close,\n            bars_to_repair=e.best_in_window_bars,\n            repair_fraction=e.best_in_window,\n        )\n        for e in scan_episodes(bars, cfg) if e.fired\n    ]\n'
SOURCES['leveraged'] = '"""台灣50正2（00631L）的價格模型。\n\n實務上訊號來自加權指數，執行卻在槓桿 ETF 上，兩者不是同一條線：\n\n* 00631L 追蹤的是「台灣50指數單日報酬 2 倍」，不是加權指數，\n  但兩者日報酬相關性長期在 0.95 以上，訊號層面可互用。\n* 2 倍是「單日」複製，路徑相依：盤整盤會有波動耗損，\n  單邊上漲則會優於 2 倍。\n* 內扣（管理費 + 期貨轉倉/避險）約年化 1%～1.5%，逐日侵蝕淨值。\n\n沒有實際 00631L 日線時，用本模組由指數日報酬合成一條可回測的淨值路徑；\n有實際日線就直接餵真實價格，模型只用來做對照。\n"""\n\nfrom __future__ import annotations\n\nfrom .bars import Bar\nfrom .config import CostConfig\n\n\ndef synth_leveraged_path(\n    bars: list[Bar],\n    cost: CostConfig,\n    leverage: float = 2.0,\n    start_price: float = 100.0,\n) -> list[float]:\n    """由指數日線合成 2 倍槓桿 ETF 的收盤淨值序列。"""\n    path = [start_price]\n    for prev, cur in zip(bars, bars[1:]):\n        r = cur.close / prev.close - 1.0\n        nav = path[-1] * (1.0 + leverage * r - cost.daily_carry)\n        # 槓桿 ETF 淨值不會歸零/轉負，但單日 -50% 指數已超出任何現實情境；\n        # 保底避免回測數值爆掉。\n        path.append(max(nav, 1e-6))\n    return path\n\n\ndef decay_estimate(index_return: float, realized_vol: float, days: int, cost: CostConfig,\n                   leverage: float = 2.0) -> float:\n    """粗估持有 N 日後，槓桿 ETF 相對「指數報酬 × 2」的落差。\n\n    近似式: 2x 報酬 ≈ L·r - 0.5·L·(L-1)·σ²·(days/252) - carry·days/252\n    用來提醒：這個劇本必須是「快速、單邊」的行情才值得用槓桿工具。\n    """\n    variance_drag = 0.5 * leverage * (leverage - 1.0) * (realized_vol ** 2) * (days / cost.trading_days)\n    carry_drag = cost.daily_carry * days\n    return leverage * index_return - variance_drag - carry_drag\n'
SOURCES['engine'] = '"""策略執行引擎（狀態機）。\n\n判斷一律用收盤價，成交一律落在下一個交易日，\n避免「當日收盤發訊號、當日收盤成交」這種實務上做不到的假設。\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom datetime import date\n\nfrom .bars import Bar\nfrom .config import DEFAULT_CONFIG, StrategyConfig\nfrom .levels import Levels, build_levels\nfrom .setup import FastRepairSetup, detect_setups\n\n\n@dataclass\nclass Fill:\n    d: date\n    side: str            # buy / sell\n    reason: str\n    index_price: float\n    etf_price: float\n    weight: float        # 買：佔訊號日權益比例；賣：佔當時持股比例\n    units: float\n    cash_flow: float\n\n    def describe(self) -> str:\n        verb = "買進" if self.side == "buy" else "賣出"\n        return (f"{self.d} {verb} {self.weight:>6.1%} @ ETF {self.etf_price:,.2f} "\n                f"(指數 {self.index_price:,.0f})  {self.reason}")\n\n\n@dataclass\nclass Trade:\n    setup: FastRepairSetup\n    levels: Levels\n    target_weight: float\n    fills: list[Fill] = field(default_factory=list)\n    entry_date: date | None = None\n    exit_date: date | None = None\n    exit_reason: str = "open"\n    equity_at_entry: float = 1.0\n    equity_at_exit: float = 1.0\n    max_adverse_index: float = 0.0   # 進場後指數最大不利波動\n    reached_prior_high: bool = False\n    swing_high: float = 0.0          # 訊號後的最高收盤（回檔梯的基準）\n    pending_ladder: list[tuple[float, float]] = field(default_factory=list)  # 尚未成交的加碼梯\n    bars_held: int = 0\n\n    @property\n    def ret(self) -> float:\n        return self.equity_at_exit / self.equity_at_entry - 1.0\n\n    @property\n    def filled_weight(self) -> float:\n        return sum(f.weight for f in self.fills if f.side == "buy")\n\n\n@dataclass\nclass Result:\n    trades: list[Trade]\n    equity_curve: list[tuple[date, float]]\n    setups: list[FastRepairSetup]\n\n    @property\n    def final_equity(self) -> float:\n        return self.equity_curve[-1][1] if self.equity_curve else 1.0\n\n\ndef _blended_stop_distance(entry_index: float, levels: Levels, cfg: StrategyConfig) -> float:\n    """兩段式停損的預期虧損距離（指數口徑）。\n\n    先在警戒線減碼一半，剩下的在谷底出清，真正的預期損失介於兩者之間。\n    警戒段用 `levels.stop_line`（含主防線的假跌破緩衝）—— 那才是真正會成交的價位。\n    """\n    to_warn = max(entry_index - levels.stop_line, 0.0) / entry_index\n    to_trough = max(entry_index - levels.invalidation, 0.0) / entry_index\n    f = cfg.exit.warn_derisk_fraction\n    blended = f * to_warn + (1.0 - f) * to_trough\n    return max(blended, cfg.sizing.min_stop_distance)\n\n\ndef position_size(entry_index: float, levels: Levels, cfg: StrategyConfig) -> float:\n    """這一筆交易的目標持股水位（佔權益比例）。\n\n    槓桿 ETF 的虧損 ≈ 指數跌幅 × 2，所以指數停損距離越遠、部位必須越小。\n    """\n    expected_loss = cfg.sizing.leverage * _blended_stop_distance(entry_index, levels, cfg)\n    return min(cfg.sizing.max_weight, cfg.sizing.risk_per_trade / expected_loss)\n\n\ndef moving_average(bars: list[Bar], period: int) -> list[float | None]:\n    """加權指數收盤的簡單移動平均；資料不足時為 None。"""\n    out: list[float | None] = []\n    total = 0.0\n    for i, b in enumerate(bars):\n        total += b.close\n        if i >= period:\n            total -= bars[i - period].close\n        out.append(total / period if i >= period - 1 else None)\n    return out\n\n\ndef breakout_stop(cfg: StrategyConfig, prior_high: float, peak_since_breakout: float,\n                  ma: float | None) -> tuple[float, str]:\n    """創高之後的出場線，回傳 (價位, 說明)。"""\n    mode = cfg.exit.exit_mode\n    trail = peak_since_breakout * (1 - cfg.exit.trail_drawdown)\n\n    if mode == "trail":\n        return trail, f"移動停利：自 {peak_since_breakout:,.0f} 回檔 {cfg.exit.trail_drawdown:.0%}"\n\n    # 均線棘輪：先以前高為出場線，等 MA 爬過前高才改看 MA\n    if ma is None or ma <= prior_high:\n        ratchet, why = prior_high, f"跌破前高 {prior_high:,.0f}（MA{cfg.exit.ma_period} 尚未站上）"\n    else:\n        ratchet, why = ma, f"跌破 MA{cfg.exit.ma_period} {ma:,.0f}"\n\n    if mode == "ma_ratchet":\n        return ratchet, why\n    if mode == "both":\n        return ((trail, f"移動停利：自 {peak_since_breakout:,.0f} 回檔 {cfg.exit.trail_drawdown:.0%}")\n                if trail > ratchet else (ratchet, why))\n    raise ValueError(f"未知的 exit_mode: {mode!r}")\n\n\ndef ladder_weights(cfg: StrategyConfig, target: float) -> list[tuple[float, float]]:\n    """回檔加碼梯的實際權重。\n\n    底倉與加碼梯合計必須剛好等於目標水位：加碼梯的設定值是「彼此之間的比例」，\n    實際可用的額度是 1 − base_weight。少了這一步，base_weight=1.0 會在滿倉之後\n    再加 60%，把部位推到目標的 160%，直接突破風險預算。\n    """\n    remaining = max(0.0, 1.0 - cfg.entry.base_weight)\n    total = sum(w for _, w in cfg.entry.pullback_ladder)\n    if remaining <= 0 or total <= 0:\n        return []\n    return [(thr, w / total * remaining * target) for thr, w in cfg.entry.pullback_ladder]\n\n\ndef _risk_scale(entry_index: float, fill_index: float, levels: Levels,\n                cfg: StrategyConfig) -> float:\n    """往上加碼時的權重縮放。\n\n    部位大小是在訊號日算好的；若之後在更高的價位補倉，同樣的股數要承擔更長的\n    停損距離，實際風險就會超出預算。這裡按停損距離等比縮小，並限制在 1.0 以內\n    ——只會因為買貴而縮手，不會因為買便宜而放大部位。\n    """\n    base = _blended_stop_distance(entry_index, levels, cfg)\n    now = _blended_stop_distance(fill_index, levels, cfg)\n    return min(1.0, base / now) if now > 0 else 1.0\n\n\nclass Engine:\n    """單一部位、單一標的（台灣50正2）的回測 / 實盤訊號引擎。"""\n\n    def __init__(self, cfg: StrategyConfig | None = None):\n        self.cfg = cfg or DEFAULT_CONFIG\n\n    def run(self, bars: list[Bar], etf_prices: list[float]) -> Result:\n        cfg = self.cfg\n        if len(bars) != len(etf_prices):\n            raise ValueError("指數日線與 ETF 價格長度不一致")\n\n        setups = detect_setups(bars, cfg.setup)\n        by_trigger = {s.trigger_index: s for s in setups}\n        mas = (moving_average(bars, cfg.exit.ma_period)\n               if cfg.exit.exit_mode in ("ma_ratchet", "both") else [None] * len(bars))\n\n        cash, units = 1.0, 0.0\n        equity_curve: list[tuple[date, float]] = []\n        trades: list[Trade] = []\n\n        trade: Trade | None = None\n        pending: list[tuple[str, float, str]] = []   # (side, weight, reason)\n        base_equity = 1.0\n        swing_high = 0.0\n        peak_since_breakout = 0.0\n        unfilled: list[tuple[float, float]] = []\n        derisked = reloaded = took_profit = False\n\n        for i, bar in enumerate(bars):\n            px = etf_prices[i]\n\n            # ---- 1. 執行前一交易日收盤掛出的委託 ----\n            for side, weight, reason in pending:\n                if side == "buy":\n                    notional = weight * base_equity\n                    if notional <= 0:\n                        continue\n                    fee = notional * cfg.cost.buy_cost\n                    u = notional / px\n                    cash -= notional + fee\n                    units += u\n                    assert trade is not None\n                    trade.fills.append(\n                        Fill(bar.d, "buy", reason, bar.close, px, weight, u, -(notional + fee))\n                    )\n                    if trade.entry_date is None:\n                        trade.entry_date = bar.d\n                        trade.equity_at_entry = base_equity\n                else:\n                    u = units * weight\n                    if u <= 0:\n                        continue\n                    proceeds = u * px * (1 - cfg.cost.sell_cost)\n                    cash += proceeds\n                    units -= u\n                    assert trade is not None\n                    trade.fills.append(\n                        Fill(bar.d, "sell", reason, bar.close, px, weight, u, proceeds)\n                    )\n            pending = []\n            equity = cash + units * px\n\n            # ---- 2. 收盤後評估，掛出明日委託 ----\n            if trade is None:\n                setup = by_trigger.get(i)\n                if setup is not None:\n                    levels = build_levels(setup.peak, setup.trough, cfg.levels)\n                    target = position_size(bar.close, levels, cfg)\n                    trade = Trade(setup=setup, levels=levels, target_weight=target,\n                                  equity_at_entry=equity)\n                    base_equity = equity\n                    swing_high = bar.close\n                    peak_since_breakout = 0.0\n                    derisked = reloaded = took_profit = False\n                    unfilled = ladder_weights(cfg, target)\n                    pending.append(("buy", cfg.entry.base_weight * target, "底倉：訊號確認，不等回檔"))\n            else:\n                lv, c = trade.levels, bar.close\n                swing_high = max(swing_high, c)\n                bars_since = i - trade.setup.trigger_index\n                trade.swing_high = swing_high\n                trade.bars_held = bars_since\n                if trade.fills:\n                    trade.max_adverse_index = min(\n                        trade.max_adverse_index, c / trade.fills[0].index_price - 1.0\n                    )\n                zone = lv.zone(c)\n\n                if zone == "invalidated" and cfg.exit.hard_stop_at_trough:\n                    unfilled = []\n                    pending.append(("sell", 1.0, f"劇本失效：收盤跌破谷底 {lv.invalidation:,.0f}"))\n                    trade.exit_reason = "stop_trough"\n\n                elif zone == "warning" and not derisked:\n                    unfilled = []\n                    derisked = True\n                    verb = "清倉" if cfg.exit.warn_derisk_fraction >= 1.0 else "減碼"\n                    why = (f"{cfg.levels.warn_line_ratio:.1%} 回補位"\n                           if lv.stop_line >= lv.warn_line\n                           else f"主防線 {lv.half_line:,.0f} 的 "\n                                f"{cfg.levels.half_line_buffer:.1%} 緩衝")\n                    pending.append(("sell", cfg.exit.warn_derisk_fraction,\n                                    f"警戒{verb}：收盤跌破 {lv.stop_line:,.0f}（{why}）"))\n\n                else:\n                    adds_allowed = zone in ("healthy", "buffer", "breakout")\n\n                    if c >= lv.peak:\n                        trade.reached_prior_high = True\n                        peak_since_breakout = max(peak_since_breakout, c)\n\n                    # (a) 突破前高 → 依設定補齊或取消未成交的分批單\n                    breakout_filled = False\n                    if trade.reached_prior_high and unfilled:\n                        if cfg.entry.breakout_fills_remainder and adds_allowed:\n                            scale = _risk_scale(trade.setup.trigger_close, c, lv, cfg)\n                            for _, w in unfilled:\n                                pending.append(("buy", w * scale,\n                                                f"突破補齊：站上前高 {lv.peak:,.0f}"\n                                                f"（風險縮放 {scale:.0%}）"))\n                            breakout_filled = True\n                        unfilled = []\n\n                    # (b) 回到前高：可選的分批落袋（預設 0，前高不是賣出的理由）\n                    if (trade.reached_prior_high and not took_profit and units > 0\n                            and not breakout_filled and cfg.exit.target_take_fraction > 0):\n                        took_profit = True\n                        pending.append(("sell", cfg.exit.target_take_fraction,\n                                        f"目標達陣：回到前高 {lv.peak:,.0f}"))\n\n                    # (c) 創高之後，出場全部交給停利機制\n                    if trade.reached_prior_high and units > 0:\n                        stop, why = breakout_stop(cfg, lv.peak, peak_since_breakout, mas[i])\n                        if c < stop:\n                            pending.append(("sell", 1.0, why))\n                            trade.exit_reason = "trail"\n\n                    # (d) 減碼後收復主防線 → 補回一次\n                    if derisked and not reloaded and adds_allowed and c >= lv.half_line and units > 0:\n                        reloaded = True\n                        # 解除減碼旗標：回補之後若再度跌破警戒線，還要能再減一次\n                        derisked = False\n                        pending.append(("buy", trade.filled_weight * cfg.exit.warn_derisk_fraction,\n                                        f"回補：收復主防線 {lv.half_line:,.0f}"))\n\n                    # (e) 回檔加碼梯\n                    if unfilled and adds_allowed:\n                        pullback = c / swing_high - 1.0\n                        still: list[tuple[float, float]] = []\n                        for thr, w in unfilled:\n                            if pullback <= -thr:\n                                pending.append(("buy", w, f"回檔加碼：自波段高點 {pullback:.1%}"))\n                            else:\n                                still.append((thr, w))\n                        unfilled = still\n\n                    # (f) 時間補齊：等不到回檔，不參與才是最大的風險\n                    if unfilled and adds_allowed and bars_since >= cfg.entry.fill_timeout_bars:\n                        scale = _risk_scale(trade.setup.trigger_close, c, lv, cfg)\n                        for _, w in unfilled:\n                            pending.append(("buy", w * scale,\n                                            f"時間補齊：{bars_since} 個交易日未見回檔"\n                                            f"（風險縮放 {scale:.0%}）"))\n                        unfilled = []\n\n                    # (g) 劇本過期\n                    if (not trade.reached_prior_high and units > 0\n                            and bars_since >= cfg.setup.setup_expiry_bars):\n                        unfilled = []\n                        pending.append(("sell", 1.0, f"劇本過期：{bars_since} 個交易日未創高"))\n                        trade.exit_reason = "expired"\n\n                trade.pending_ladder = list(unfilled)\n\n                # 部位歸零且沒有待買單 → 結案\n                if units <= 0 and trade.entry_date is not None and not any(\n                        s == "buy" for s, _, _ in pending):\n                    if trade.exit_reason == "open":\n                        trade.exit_reason = "flat"\n                    trade.exit_date = bar.d\n                    trade.equity_at_exit = equity\n                    trades.append(trade)\n                    trade = None\n\n            equity_curve.append((bar.d, cash + units * px))\n\n        if trade is not None:\n            trade.exit_date = bars[-1].d\n            trade.equity_at_exit = cash + units * etf_prices[-1]\n            trades.append(trade)\n\n        return Result(trades=trades, equity_curve=equity_curve, setups=setups)\n'
SOURCES['backtest'] = '"""回測與績效統計。"""\n\nfrom __future__ import annotations\n\nimport statistics\nfrom dataclasses import dataclass\n\nfrom .bars import Bar\nfrom .config import DEFAULT_CONFIG, StrategyConfig\nfrom .engine import Engine, Result\nfrom .leveraged import synth_leveraged_path\n\n\n@dataclass(frozen=True)\nclass Stats:\n    n_setups: int\n    n_trades: int\n    hit_prior_high: int\n    hit_rate: float\n    win_rate: float\n    avg_return: float\n    median_return: float\n    best: float\n    worst: float\n    total_return: float\n    max_drawdown: float\n    median_max_adverse: float\n\n    def render(self) -> str:\n        return "\\n".join([\n            f"訊號次數            {self.n_setups}",\n            f"實際交易            {self.n_trades}",\n            f"回到前高            {self.hit_prior_high} ({self.hit_rate:.0%})",\n            f"獲利比例            {self.win_rate:.0%}",\n            f"單筆平均報酬        {self.avg_return:+.1%}",\n            f"單筆中位數報酬      {self.median_return:+.1%}",\n            f"最佳 / 最差         {self.best:+.1%} / {self.worst:+.1%}",\n            f"權益總報酬          {self.total_return:+.1%}",\n            f"權益最大回檔        {self.max_drawdown:.1%}",\n            f"進場後指數最大逆行  {self.median_max_adverse:.1%}（中位數）",\n        ])\n\n\ndef align_etf(bars: list[Bar], etf_bars: list[Bar]) -> tuple[list[Bar], list[float]]:\n    """把 ETF 日線對齊到指數日線，只保留兩邊都有交易的日子。\n\n    00631L 2014-10-31 才掛牌，比加權指數短很多；用真實 ETF 價格回測時，\n    回測期間會自動縮到重疊區間，而不是拿合成價去補前面那一段。\n    """\n    etf_by_date = {b.d: b.close for b in etf_bars}\n    kept = [(b, etf_by_date[b.d]) for b in bars if b.d in etf_by_date]\n    if not kept:\n        raise ValueError("指數與 ETF 日線沒有重疊的交易日")\n    return [b for b, _ in kept], [p for _, p in kept]\n\n\ndef run_backtest(bars: list[Bar], cfg: StrategyConfig | None = None,\n                 etf_prices: list[float] | None = None) -> tuple[Result, Stats]:\n    cfg = cfg or DEFAULT_CONFIG\n    if etf_prices is None:\n        etf_prices = synth_leveraged_path(bars, cfg.cost, cfg.sizing.leverage)\n    result = Engine(cfg).run(bars, etf_prices)\n    return result, summarize(result)\n\n\ndef summarize(result: Result) -> Stats:\n    rets = [t.ret for t in result.trades]\n    hits = sum(1 for t in result.trades if t.reached_prior_high)\n    adverse = [t.max_adverse_index for t in result.trades] or [0.0]\n\n    peak = -1e18\n    max_dd = 0.0\n    for _, eq in result.equity_curve:\n        peak = max(peak, eq)\n        max_dd = min(max_dd, eq / peak - 1.0)\n\n    n = len(result.trades)\n    return Stats(\n        n_setups=len(result.setups),\n        n_trades=n,\n        hit_prior_high=hits,\n        hit_rate=hits / n if n else 0.0,\n        win_rate=sum(1 for r in rets if r > 0) / n if n else 0.0,\n        avg_return=statistics.fmean(rets) if rets else 0.0,\n        median_return=statistics.median(rets) if rets else 0.0,\n        best=max(rets) if rets else 0.0,\n        worst=min(rets) if rets else 0.0,\n        total_return=result.final_equity - 1.0,\n        max_drawdown=max_dd,\n        median_max_adverse=statistics.median(adverse),\n    )\n\n\n@dataclass(frozen=True)\nclass TradeReview:\n    """單筆 ETF 交易的完整檢視：報酬、期間極值，以及當初的進場條件。"""\n\n    trade: "object"          # engine.Trade\n    bars_held: int\n    weight: float            # 實際建立的部位水位（佔權益比例）\n    stop_distance: float     # 訊號日收盤到停損線的距離\n    mfe: float               # 期間最大浮動獲利（權益，相對進場）\n    mae: float               # 期間最大浮動虧損（權益，相對進場）\n    max_drawdown: float      # 期間內從波段高點起算的最大回撤\n\n    @property\n    def setup(self):\n        return self.trade.setup\n\n    @property\n    def levels(self):\n        return self.trade.levels\n\n    @property\n    def from_peak(self) -> float:\n        s = self.setup\n        return s.trigger_close / s.peak - 1.0\n\n    @property\n    def exit_reason(self) -> str:\n        t = self.trade\n        if t.exit_reason == "open":\n            return "尚未出場（持有中）"\n        fills = t.fills\n        if len(fills) > 1 and fills[-1].side == "sell":\n            return fills[-1].reason\n        return t.exit_reason\n\n\ndef review_trades(result: Result) -> list[TradeReview]:\n    """把權益曲線切成逐筆部位，算出每筆的 MFE / MAE / 期間最大回撤。\n\n    極值以**整體權益**衡量 —— 沒滿倉的時候現金部位會稀釋波動，\n    這正是實際帳戶看到的數字，不是標的本身的漲跌。\n    """\n    idx = {d: i for i, (d, _) in enumerate(result.equity_curve)}\n    out: list[TradeReview] = []\n    for t in result.trades:\n        if t.entry_date is None:\n            continue\n        a = idx[t.entry_date]\n        b = idx[t.exit_date] if t.exit_date in idx else len(result.equity_curve) - 1\n        seg = [eq for _, eq in result.equity_curve[a:b + 1]]\n        base = t.equity_at_entry\n        if not seg or base <= 0:\n            continue\n        peak, dd = seg[0], 0.0\n        for x in seg:\n            peak = max(peak, x)\n            dd = min(dd, x / peak - 1.0)\n        entry = t.setup.trigger_close\n        out.append(TradeReview(\n            trade=t, bars_held=b - a, weight=t.filled_weight,\n            stop_distance=max(entry - t.levels.stop_line, 0.0) / entry,\n            mfe=max(seg) / base - 1.0, mae=min(seg) / base - 1.0,\n            max_drawdown=dd))\n    return out\n'
SOURCES['plan'] = '"""把訊號翻譯成可以直接下單的操作計畫。"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom .config import DEFAULT_CONFIG, StrategyConfig\nfrom .engine import ladder_weights, position_size\nfrom .levels import Levels, build_levels\n\n\n@dataclass(frozen=True)\nclass Order:\n    tag: str\n    weight: float          # 佔總權益比例\n    trigger: str\n    index_level: float | None   # 觸發時的指數概略位置（回檔梯以波段高點推算）\n\n\n@dataclass(frozen=True)\nclass TradePlan:\n    levels: Levels\n    reference_index: float\n    target_weight: float\n    orders: list[Order]\n    cfg: StrategyConfig\n    capital: float | None = None\n\n    def render(self) -> str:\n        lv = self.levels\n        out: list[str] = []\n        out.append("=" * 68)\n        out.append("台灣指數快速修復 → 台灣50正2(00631L) 回檔入場計畫")\n        out.append("=" * 68)\n        out.append("")\n        out.append(f"前波高點 P        {lv.peak:>10,.0f}   ← 主目標（歷史 89% 會回到這裡）")\n        out.append(f"波段谷底 T        {lv.trough:>10,.0f}   ← 失效線：收盤跌破 = 全部出場")\n        out.append(f"跌幅 R            {lv.drop:>10,.0f} 點 ({lv.drop_pct:.1%})")\n        out.append("")\n        ex, lc = self.cfg.exit, self.cfg.levels\n        derisk = "全部出場" if ex.warn_derisk_fraction >= 1.0 else f"減碼 {ex.warn_derisk_fraction:.0%}"\n        out.append("關鍵價位")\n        out.append(f"  61.8% 回補      {lv.trough + 0.618 * lv.drop:>10,.0f}")\n        if abs(lc.warn_line_ratio - lc.half_line_ratio) < 1e-9:\n            # 警戒線與主防線重合：只印一條，避免看起來像兩個不同的動作\n            out.append(f"  主防線 = 警戒線 {lv.half_line:>10,.0f}   "\n                       f"({lc.half_line_ratio:.1%} 回補位) 收盤跌破 → {derisk}")\n            out.append(f"  容忍緩衝 (-{lc.half_line_buffer:.1%}){lv.half_line_with_buffer:>8,.0f}   "\n                       f"假跌破區，不動作")\n        else:\n            out.append(f"  主防線 ({lc.half_line_ratio:.0%})    {lv.half_line:>10,.0f}   歷史回檔多數止步於此")\n            out.append(f"  容忍緩衝 (-{lc.half_line_buffer:.1%}){lv.half_line_with_buffer:>8,.0f}   "\n                       f"假跌破區，不加碼也不減碼")\n            out.append(f"  警戒線 ({lc.warn_line_ratio:.1%}) {lv.warn_line:>10,.0f}   收盤跌破 → {derisk}")\n        out.append(f"  失效線          {lv.invalidation:>10,.0f}   收盤跌破 → 清倉")\n        out.append("")\n        out.append(f"現價（參考）      {self.reference_index:>10,.0f}   "\n                   f"補回 {lv.repair_fraction(self.reference_index):.0%}，"\n                   f"距前高 {self.reference_index / lv.peak - 1:+.1%}，分區：{lv.zone(self.reference_index)}")\n        out.append("")\n        out.append(f"目標總持股水位    {self.target_weight:.0%} 的權益"\n                   + (f"（約 {self.capital * self.target_weight:,.0f} 元）" if self.capital else ""))\n        out.append("")\n        out.append("進場梯（權重為佔總權益比例）"\n                   if len(self.orders) > 1 else "進場（權重為佔總權益比例）")\n        for o in self.orders:\n            if o.weight > 0:\n                size = f"{o.weight:>6.1%}"\n                amount = f"  ≈ {self.capital * o.weight:,.0f} 元" if self.capital else ""\n            else:\n                size, amount = "  剩餘", ""\n            level = f"（指數約 {o.index_level:,.0f}）" if o.index_level else ""\n            out.append(f"  {o.tag:<10} {size}{amount:<18} {o.trigger}{level}")\n        out.append("")\n        out.append("出場")\n        if ex.target_take_fraction > 0:\n            out.append(f"  觸及 {lv.peak:,.0f}    → 賣出 {ex.target_take_fraction:.0%} 落袋，其餘轉停利")\n        else:\n            out.append(f"  觸及 {lv.peak:,.0f}    → 不賣（前高不是賣出的理由），啟動停利")\n        if ex.exit_mode == "trail":\n            out.append(f"  移動停利          自波段最高收盤回檔 {ex.trail_drawdown:.0%}（指數）→ 出清")\n        elif ex.exit_mode == "ma_ratchet":\n            out.append(f"  均線棘輪          出場線 = max(前高, MA{ex.ma_period})，跌破 → 出清")\n        else:\n            out.append(f"  停利（取較緊）    max(前高, MA{ex.ma_period}) 與 "\n                       f"自高點回檔 {ex.trail_drawdown:.0%} 兩者取高 → 跌破出清")\n        out.append(f"  收盤 < {lv.stop_line:,.0f}   → {derisk}")\n        out.append(f"  收盤 < {lv.invalidation:,.0f}   → 全部出場，不留倉")\n        out.append("")\n        out.append("提醒：00631L 為 2 倍槓桿，指數 -1% ≈ ETF -2%（另有波動耗損與內扣）。")\n        to_stop = lv.invalidation / self.reference_index - 1\n        to_warn = lv.stop_line / self.reference_index - 1\n        out.append(f"      現價到停損線 {to_warn:.1%}（ETF 約 {2 * to_warn:.0%}）、"\n                   f"到失效線 {to_stop:.1%}（ETF 約 {2 * to_stop:.0%}）。")\n        return "\\n".join(out)\n\n\ndef build_plan(peak: float, trough: float, reference_index: float,\n               cfg: StrategyConfig | None = None,\n               capital: float | None = None) -> TradePlan:\n    cfg = cfg or DEFAULT_CONFIG\n    lv = build_levels(peak, trough, cfg.levels)\n    target = position_size(reference_index, lv, cfg)\n\n    orders = [Order(tag="底倉", weight=cfg.entry.base_weight * target,\n                    trigger="訊號確認後次一交易日市價買進，不等回檔", index_level=None)]\n    for thr, w in ladder_weights(cfg, target):\n        orders.append(\n            Order(tag=f"回檔 -{thr:.0%}", weight=w,\n                  trigger=f"自訊號後波段最高收盤回檔 {thr:.0%} 且收盤仍在 {lv.half_line_with_buffer:,.0f} 之上",\n                  index_level=reference_index * (1 - thr))\n        )\n    if len(orders) > 1:      # 只有存在加碼梯時，時間補齊才有意義\n        orders.append(\n            Order(tag="時間補齊", weight=0.0,\n                  trigger=f"訊號後 {cfg.entry.fill_timeout_bars} 個交易日仍未觸發回檔梯 → 剩餘部位市價補齊",\n                  index_level=None)\n        )\n    return TradePlan(levels=lv, reference_index=reference_index, target_weight=target,\n                     orders=orders, cfg=cfg, capital=capital)\n'
SOURCES['status'] = '"""目前這一輪劇本的即時狀態：手上該做什麼、下一個觸發點在哪。"""\n\nfrom __future__ import annotations\n\nfrom .bars import Bar\nfrom .config import StrategyConfig\nfrom .engine import Result, Trade\n\n\ndef render_status(result: Result, bars: list[Bar], cfg: StrategyConfig,\n                  capital: float | None = None) -> str:\n    last = bars[-1]\n    out: list[str] = ["=" * 72,\n                      f"部位現況（資料截至 {last.d}，指數收盤 {last.close:,.0f}）",\n                      "=" * 72, ""]\n\n    open_trades = [t for t in result.trades if t.exit_reason == "open"]\n    if not open_trades:\n        if result.setups:\n            s = result.setups[-1]\n            out.append("目前沒有進行中的部位。最近一次訊號：")\n            out.append(f"  {s.describe()}")\n            if result.trades:\n                t = result.trades[-1]\n                out.append(f"  該筆已於 {t.exit_date} 出場（{t.exit_reason}），報酬 {t.ret:+.1%}")\n        else:\n            out.append("目前沒有進行中的部位，資料期間內也沒有觸發過訊號。")\n        out.append("")\n        out.append("下一次進場條件：自高點回檔 ≥"\n                   f"{cfg.setup.min_drawdown:.0%} 後，谷底起 ≤{cfg.setup.max_repair_bars} 個交易日內"\n                   f"收盤補回 ≥{cfg.setup.repair_fraction:.0%}。")\n        return "\\n".join(out)\n\n    t = open_trades[-1]\n    lv = t.levels\n    c = last.close\n    zone = lv.zone(c)\n\n    out.append(f"訊號        {t.setup.describe()}")\n    out.append(f"已持有      {t.bars_held} 個交易日"\n               f"（劇本有效期 {cfg.setup.setup_expiry_bars} 日）")\n    out.append("")\n    out.append(f"目標水位    {t.target_weight:.1%}"\n               + (f"（約 {capital * t.target_weight:,.0f} 元）" if capital else ""))\n    out.append(f"已建立      {t.filled_weight:.1%}"\n               + (f"（約 {capital * t.filled_weight:,.0f} 元）" if capital else "")\n               + f"\u3000＝ 目標的 {t.filled_weight / t.target_weight:.0%}")\n    out.append("")\n    out.append("已成交")\n    for f in t.fills:\n        out.append("  " + f.describe())\n    out.append("")\n\n    zone_note = {\n        "breakout": "已站上前高 —— 出場交給移動停利",\n        "healthy": "劇本正常 —— 回檔就是加碼機會",\n        "buffer": "主防線假跌破容忍區 —— 不加碼、也不減碼",\n        "caution": "已跌破主防線 —— 停止加碼，只留現有部位",\n        "warning": "已跌破警戒線 —— 減碼一半",\n        "invalidated": "已跌破谷底 —— 劇本失效，清倉",\n    }[zone]\n    out.append(f"目前分區    {zone}\u3000{zone_note}")\n    out.append(f"距前高      {c / lv.peak - 1:+.1%}\u3000"\n               f"距主防線 {c / lv.half_line - 1:+.1%}\u3000"\n               f"距停損線 {c / lv.stop_line - 1:+.1%}\u3000"\n               f"距失效線 {c / lv.invalidation - 1:+.1%}")\n    out.append("")\n\n    out.append("下一步（收盤價判定，次一交易日執行）")\n    if t.pending_ladder:\n        for thr, w in t.pending_ladder:\n            trigger_px = t.swing_high * (1 - thr)\n            gap = trigger_px / c - 1\n            blocked = trigger_px < lv.half_line_with_buffer\n            note = "\u3000⚠ 觸發價已低於假跌破緩衝線，屆時不執行" if blocked else ""\n            amount = f"（約 {capital * w:,.0f} 元）" if capital else ""\n            out.append(f"  買 {w:>6.1%}{amount}\u3000收盤 ≤ {trigger_px:,.0f}"\n                       f"（自波段高 {t.swing_high:,.0f} 回檔 {thr:.0%}，距現價 {gap:+.1%}）{note}")\n        remaining = cfg.entry.fill_timeout_bars - t.bars_held\n        if remaining > 0:\n            out.append(f"  買 {\'剩餘\':>6}\u3000再過 {remaining} 個交易日仍未觸發回檔梯 → 市價補齊")\n        else:\n            out.append(f"  買 {\'剩餘\':>6}\u3000已過時間門檻 → 次一交易日市價補齊")\n        if cfg.entry.breakout_fills_remainder:\n            out.append(f"  買 {\'剩餘\':>6}\u3000收盤 > {lv.peak:,.0f}（前高）→ 補齊，權重按停損距離縮放")\n    else:\n        out.append("  加碼梯已全部處理，不再加碼")\n\n    if cfg.exit.target_take_fraction > 0:\n        out.append(f"  賣 {cfg.exit.target_take_fraction:>6.0%}\u3000收盤 ≥ {lv.peak:,.0f}（前高）")\n    if t.reached_prior_high:\n        out.append(f"  賣 {\'全部\':>6}\u3000自波段最高收盤回檔 {cfg.exit.trail_drawdown:.0%}（移動停利已啟動）")\n    else:\n        out.append(f"  賣 {\'全部\':>6}\u3000創高後啟動移動停利（自最高收盤回檔 {cfg.exit.trail_drawdown:.0%}）")\n    note = ("警戒線" if lv.stop_line >= lv.warn_line\n            else f"主防線 {lv.half_line:,.0f} 的 {cfg.levels.half_line_buffer:.1%} 緩衝")\n    out.append(f"  賣 {cfg.exit.warn_derisk_fraction:>6.0%}\u3000收盤 < {lv.stop_line:,.0f}（{note}）")\n    out.append(f"  賣 {\'全部\':>6}\u3000收盤 < {lv.invalidation:,.0f}（失效線）")\n    out.append("")\n\n    # 實際會把部位清光的第一條線：警戒線設定為全數出場時就是它，否則才是失效線\n    full_exit = (lv.stop_line if cfg.exit.warn_derisk_fraction >= 1.0 else lv.invalidation)\n    lev = cfg.sizing.leverage\n    gap = full_exit / c - 1\n    out.append(f"預期最大損失  跌到出清線 {full_exit:,.0f}（{gap:.1%}），"\n               f"00631L 約 {lev * gap:.0%}；依已建立的 {t.filled_weight:.1%} 部位，"\n               f"權益衝擊約 {t.filled_weight * lev * gap:.1%}")\n    if full_exit != lv.invalidation:\n        gap2 = lv.invalidation / c - 1\n        out.append(f"              （跳空直接摜破失效線 {lv.invalidation:,.0f} 的極端情形："\n                   f"{gap2:.1%}，權益衝擊約 {t.filled_weight * lev * gap2:.1%}）")\n    return "\\n".join(out)\n\n\ndef open_trade(result: Result) -> Trade | None:\n    for t in reversed(result.trades):\n        if t.exit_reason == "open":\n            return t\n    return None\n'
SOURCES['futures'] = '"""台指期執行版：連續合約、換倉、固定口數的槓桿部位。\n\n與 ETF 版的三個差異\n-------------------\n1. **換倉**：近月合約到期要換到次月。換倉當日以「近月收盤」平倉、\n   「次月同日收盤」建倉，兩者的價差**不計入損益** —— 那是逆價差/正價差，\n   不是賺賠。`build_continuous()` 就是在做這件事。\n\n2. **槓桿由停損距離決定**：ETF 的槓桿是固定的（00631L 為 2 倍），期貨可以自己選。\n   同樣的風險預算下，停損越近就能放越大：\n\n       槓桿 L = 風險預算 ÷ 停損距離，上限 `max_leverage`\n\n   注意這裡**不套用 `min_stop_distance` 下限** —— 那個下限是為了防止\n   固定槓桿的 ETF 算出過大的水位；期貨改由槓桿上限承擔同樣的角色。\n   若保留下限，L 會恆等於 8%/5% = 1.6 倍，槓桿上限永遠碰不到。\n\n3. **進場後不調整口數**：權益變動時實際槓桿會自然漂移（賺錢時下降）。\n   因此持有期間的權益是**線性**於期貨報酬，不是複利：\n\n       權益(t) / 權益(進場) = 1 + L × (F(t)/F(進場) − 1)\n\n4. **當日成交**：加權指數 13:30 收盤、台指期 13:45 收盤，中間有 15 分鐘。\n   訊號以指數收盤判定後，還來得及在**當天的期貨收盤**成交，不必等到隔天。\n   ETF 版沒有這個空間（同一時間收盤），只能次日成交。\n   這個差別對停損特別關鍵 —— 隔夜跳空正是把「假設停損 1.5%」放大成\n   「實際虧損 7.8%」的主因。以 `entries_from_trades(same_day=...)` 切換。\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom datetime import date\n\nfrom .config import StrategyConfig\nfrom .levels import Levels\n\n#: 台指期（TX）每點新台幣，小台（MTX）為 50\nTX_POINT_VALUE = 200.0\n\n\n@dataclass(frozen=True)\nclass FuturesCost:\n    """期貨交易成本（以契約金額比例表示，單邊）。"""\n\n    #: 期交稅：契約金額的十萬分之二\n    tax_rate: float = 0.00002\n    #: 手續費，每口新台幣。換算成比例時需要指數點位與每點價值\n    commission_per_lot: float = 50.0\n    point_value: float = TX_POINT_VALUE\n\n    def one_way_rate(self, index_level: float) -> float:\n        """單邊成本佔契約金額的比例。"""\n        notional = index_level * self.point_value\n        return self.tax_rate + self.commission_per_lot / notional\n\n\ndef build_continuous(dates: list[date], front_close: list[float],\n                     contract: list[str],\n                     next_close_on_roll: dict[date, float]) -> list[float]:\n    """把近月連續序列接成「已還原換倉價差」的可交易序列。\n\n    Args:\n        dates / front_close / contract: 逐日的日期、近月收盤、該日所屬合約月。\n        next_close_on_roll: {換倉日: 次月合約在該日的收盤}。換倉日指的是\n            **舊合約的最後一天**，也就是 `contract` 即將改變的前一天。\n\n    回傳與輸入等長的序列，起點對齊 `front_close[0]`。\n\n    換倉日隔天的報酬改用「次日近月收盤 ÷ 次月在換倉日的收盤」計算，\n    價差因此不落入損益。缺換倉價差資料時退回原始跳動，並在\n    `missing_rolls()` 中可查出是哪幾天。\n    """\n    if not dates:\n        return []\n    out = [front_close[0]]\n    for i in range(1, len(dates)):\n        prev_is_roll = contract[i] != contract[i - 1]\n        base = front_close[i - 1]\n        if prev_is_roll:\n            base = next_close_on_roll.get(dates[i - 1], front_close[i - 1])\n        out.append(out[-1] * (front_close[i] / base) if base else out[-1])\n    return out\n\n\ndef missing_rolls(dates: list[date], contract: list[str],\n                  next_close_on_roll: dict[date, float]) -> list[date]:\n    """列出缺少次月報價、只能沿用原始跳動的換倉日。"""\n    return [dates[i - 1] for i in range(1, len(dates))\n            if contract[i] != contract[i - 1] and dates[i - 1] not in next_close_on_roll]\n\n\ndef futures_leverage(entry_index: float, levels: Levels, cfg: StrategyConfig,\n                     max_leverage: float = 5.0) -> float:\n    """依停損距離決定槓桿倍數，上限 `max_leverage`。\n\n    停損距離的定義與 ETF 版一致（依 `warn_derisk_fraction` 對停損線與失效線加權，\n    停損線含主防線的假跌破緩衝 —— 用警戒線會低估距離、把槓桿放得過大），\n    但**不套用 `min_stop_distance` 下限**，理由見模組說明。\n    """\n    to_warn = max(entry_index - levels.stop_line, 0.0) / entry_index\n    to_trough = max(entry_index - levels.invalidation, 0.0) / entry_index\n    f = cfg.exit.warn_derisk_fraction\n    dist = f * to_warn + (1.0 - f) * to_trough\n    if dist <= 0:\n        return max_leverage\n    return min(max_leverage, cfg.sizing.risk_per_trade / dist)\n\n\n@dataclass(frozen=True)\nclass FuturesTrade:\n    entry_date: date\n    exit_date: date | None\n    entry_index: float\n    entry_futures: float\n    exit_futures: float | None\n    leverage: float\n    stop_distance: float\n    futures_return: float    # 期貨本身的報酬\n    ret: float               # 權益報酬（已扣進出場成本）\n    #: step-down 之後的槓桿（沒有觸發時等於 leverage）\n    final_leverage: float | None = None\n    #: step-down 觸發日（沒有觸發時為 None）\n    step_date: date | None = None\n\n\n@dataclass(frozen=True)\nclass FuturesEntry:\n    """一筆待執行的期貨部位。索引對應 `dates` 的位置。"""\n\n    entry_i: int\n    exit_i: int | None       # None = 持有到資料最後一根\n    leverage: float\n    entry_index: float       # 進場當日的指數收盤\n    stop_distance: float\n    trade: object | None = None   # 產生這筆部位的引擎交易，供逐筆檢視用\n\n\ndef entries_from_trades(trades, bars, cfg: StrategyConfig,\n                        max_leverage: float = 5.0,\n                        same_day: bool = True) -> list["FuturesEntry"]:\n    """把引擎的交易轉成期貨部位。\n\n    引擎的 `Fill.d` 記的是**成交日**，而它的決策發生在前一根 K 的收盤。\n\n    Args:\n        same_day: True 代表當天期貨收盤成交（指數 13:30 收、期貨 13:45 收，\n            中間 15 分鐘足夠下單），成交索引因此往前挪一根；\n            False 則沿用引擎原本的隔日成交。\n    """\n    index_of = {b.d: i for i, b in enumerate(bars)}\n    shift = 1 if same_day else 0\n    out: list[FuturesEntry] = []\n    for t in trades:\n        if not t.fills:\n            continue\n        e_i = max(index_of[t.fills[0].d] - shift, 0)\n        x_i = (max(index_of[t.fills[-1].d] - shift, 0)\n               if len(t.fills) > 1 else None)\n        if x_i is not None and x_i <= e_i:      # 同日進出場，視為未成立\n            continue\n        entry_index = t.setup.trigger_close\n        out.append(FuturesEntry(\n            entry_i=e_i, exit_i=x_i,\n            leverage=futures_leverage(entry_index, t.levels, cfg, max_leverage),\n            entry_index=entry_index,\n            stop_distance=(entry_index - t.levels.stop_line) / entry_index,\n            trade=t))\n    return out\n\n\ndef _step_down(v: float, lev: float, anchor: float, f0: float, fut: float,\n               rate: float, target: float) -> float:\n    """把固定口數部位減到 `target` 倍：平掉多出來的名目，扣一次單邊成本。\n\n    回傳扣完成本後的權益；呼叫端把錨點重設為 (v, 今日期貨, target)。\n    """\n    cur_notional = lev * anchor * fut / f0\n    delta = max(cur_notional - target * v, 0.0)\n    return v - delta * rate\n\n\ndef vehicle_series(dates: list[date], continuous: list[float],\n                   entries: list[FuturesEntry], cost: FuturesCost,\n                   index_close: list[float],\n                   step_gain: float | None = None,\n                   step_leverage: float = 1.0) -> tuple[list[float], list[FuturesTrade]]:\n    """把「固定口數的槓桿期貨部位」攤成一條可餵給回測器的淨值序列。\n\n    空手期間持平；持有期間依 `1 + L × (F/F_entry − 1)` **線性**變動\n    （固定口數，不複利、不再平衡），並在進出場各扣一次單邊成本。\n\n    成本按契約金額計算，所以扣在權益上的比例是 `L × 單邊成本率`。\n\n    Args:\n        step_gain: **step-down**（docs/strategy.md §22）。部位權益相對進場\n            達 `1 + step_gain` 倍時（以期貨連續序列判定、當日期貨收盤成交），\n            把口數減到 `step_leverage` 倍，之後不再加回。`None` 表示關閉。\n            每筆最多觸發一次；出場日不觸發。\n    """\n    nav = [1.0] * len(dates)\n    detail: list[FuturesTrade] = []\n    by_entry = {e.entry_i: e for e in entries}\n\n    equity = 1.0\n    active: FuturesEntry | None = None\n    eq_at_entry = anchor = f0 = lev = 0.0\n    stepped = False\n    step_d: date | None = None\n\n    for i in range(len(dates)):\n        if active is None:\n            e = by_entry.get(i)\n            if e is None:\n                nav[i] = equity\n                continue\n            active = e\n            eq_at_entry = equity\n            lev = e.leverage\n            anchor = equity * (1.0 - lev * cost.one_way_rate(index_close[i]))\n            f0 = continuous[i]\n            stepped, step_d = False, None\n            nav[i] = anchor\n            continue\n\n        v = anchor * (1.0 + lev * (continuous[i] / f0 - 1.0))\n        closing = active.exit_i is not None and i == active.exit_i\n        if closing:\n            v *= (1.0 - lev * cost.one_way_rate(index_close[i]))\n        elif (step_gain is not None and not stepped\n                and v >= eq_at_entry * (1.0 + step_gain) and lev > step_leverage):\n            v = _step_down(v, lev, anchor, f0, continuous[i],\n                           cost.one_way_rate(index_close[i]), step_leverage)\n            anchor, f0, lev = v, continuous[i], step_leverage\n            stepped, step_d = True, dates[i]\n        nav[i] = v\n        if closing:\n            detail.append(FuturesTrade(\n                entry_date=dates[active.entry_i], exit_date=dates[i],\n                entry_index=active.entry_index, entry_futures=continuous[active.entry_i],\n                exit_futures=continuous[i], leverage=active.leverage,\n                stop_distance=active.stop_distance,\n                futures_return=continuous[i] / continuous[active.entry_i] - 1.0,\n                ret=v / eq_at_entry - 1.0,\n                final_leverage=lev, step_date=step_d))\n            equity, active = v, None\n\n    if active is not None:      # 持有到最後一根，未平倉\n        detail.append(FuturesTrade(\n            entry_date=dates[active.entry_i], exit_date=None,\n            entry_index=active.entry_index, entry_futures=continuous[active.entry_i],\n            exit_futures=continuous[-1], leverage=active.leverage,\n            stop_distance=active.stop_distance,\n            futures_return=continuous[-1] / continuous[active.entry_i] - 1.0,\n            ret=nav[-1] / eq_at_entry - 1.0,\n            final_leverage=lev, step_date=step_d))\n    return nav, detail\n\n\n@dataclass(frozen=True)\nclass TradeDetail:\n    """單筆期貨部位的完整檢視：報酬、期間極值，以及當初的進場條件。"""\n\n    trade: FuturesTrade\n    entry: FuturesEntry\n    bars_held: int\n    mfe: float            # 期間最大浮動獲利（權益，相對進場）\n    mae: float            # 期間最大浮動虧損（權益，相對進場）\n    max_drawdown: float   # 期間內從波段高點起算的最大回撤\n\n    # --- 進場條件（來自訊號本身） ---\n    @property\n    def setup(self):\n        return self.entry.trade.setup\n\n    @property\n    def levels(self):\n        return self.entry.trade.levels\n\n    @property\n    def exit_reason(self) -> str:\n        """出場說明。優先用出場那筆成交自己記的理由，比代碼具體。"""\n        t = self.entry.trade\n        if t.exit_reason == "open":\n            return "尚未出場（持有中）"\n        fills = t.fills\n        if len(fills) > 1 and fills[-1].side == "sell":\n            return fills[-1].reason\n        return t.exit_reason\n\n    @property\n    def from_peak(self) -> float:\n        """訊號日距離前高還有多遠（負值＝仍低於前高）。"""\n        s = self.setup\n        return s.trigger_close / s.peak - 1.0\n\n\ndef trade_details(dates: list[date], nav: list[float],\n                  entries: list[FuturesEntry],\n                  detail: list[FuturesTrade]) -> list[TradeDetail]:\n    """把淨值序列切成逐筆部位，算出每筆的 MFE / MAE / 期間最大回撤。\n\n    極值一律以**權益**衡量（已含槓桿與進場成本），所以數字就是帳戶當下看到的\n    浮動盈虧，不是期貨本身的漲跌。\n    """\n    by_entry_date = {dates[e.entry_i]: e for e in entries}\n    out: list[TradeDetail] = []\n    for t in detail:\n        e = by_entry_date.get(t.entry_date)\n        if e is None:\n            continue\n        a = e.entry_i\n        b = e.exit_i if e.exit_i is not None else len(dates) - 1\n        seg = nav[a:b + 1]\n        base = seg[0]\n        if base <= 0:\n            continue\n        peak, dd = base, 0.0\n        for x in seg:\n            peak = max(peak, x)\n            dd = min(dd, x / peak - 1.0)\n        out.append(TradeDetail(\n            trade=t, entry=e, bars_held=b - a,\n            mfe=max(seg) / base - 1.0,\n            mae=min(seg) / base - 1.0,\n            max_drawdown=dd))\n    return out\n\n\ndef format_trade_details(details: list[TradeDetail]) -> str:\n    """逐筆列出交易與當初的進場條件（中文，供 CLI 直接輸出）。"""\n    lines: list[str] = []\n    for n, d in enumerate(details, 1):\n        t, s, lv = d.trade, d.setup, d.levels\n        exit_txt = str(t.exit_date) if t.exit_date else "持有中（尚未平倉）"\n        lines.append(f"[{n}] {t.entry_date} → {exit_txt}\u3000持有 {d.bars_held} 個交易日")\n        lines.append(\n            f"    進場條件：{s.peak_date} 高點 {s.peak:,.0f} → {s.trough_date} 谷底 "\n            f"{s.trough:,.0f}（回檔 {s.drop_pct:.1%}），"\n            f"{s.bars_to_repair} 個交易日補回 {s.repair_fraction:.0%}")\n        lines.append(\n            f"              訊號日指數 {s.trigger_close:,.0f}（距前高 {d.from_peak:+.1%}）"\n            f"\u3000停損線 {lv.stop_line:,.0f}"\n            + (f"（警戒線 {lv.warn_line:,.0f} 再扣假跌破緩衝）"\n               if lv.stop_line < lv.warn_line else "（警戒線）")\n            + f"\u3000失效線 {lv.invalidation:,.0f}")\n        lines.append(\n            f"    距離停損 {t.stop_distance:.2%}\u3000→\u3000槓桿 "\n            f"{t.leverage:.2f}x（風險預算 ÷ 停損距離，上限封頂）")\n        lines.append(\n            f"    期貨報酬 {t.futures_return:+.1%}\u3000權益報酬 {t.ret:+.1%}"\n            f"\u3000最大報酬 {d.mfe:+.1%}\u3000最大不利 {d.mae:+.1%}"\n            f"\u3000期間最大回撤 {d.max_drawdown:.1%}")\n        lines.append(f"    出場原因：{d.exit_reason}")\n        lines.append("")\n    return "\\n".join(lines)\n\n\ndef trend_filter(entries: list["FuturesEntry"], bars, ma_period: int = 200,\n                 below: bool = True) -> list["FuturesEntry"]:\n    """依進場日收盤與均線的位置過濾部位。\n\n    `below=True` 只保留**收盤低於均線**的訊號。這與直覺相反，但這是逆勢策略：\n    要求「站上均線才進場」會結構性排除最深的回檔，而深回檔正是報酬最好的場景\n    （見 docs/strategy.md §18）。均線資料不足的日子一律排除。\n    """\n    from .engine import moving_average\n    ma = moving_average(bars, ma_period)\n    out = []\n    for e in entries:\n        m = ma[e.entry_i]\n        if m is None:\n            continue\n        c = bars[e.entry_i].close\n        if (c < m) if below else (c > m):\n            out.append(e)\n    return out\n\n\ndef fixed_leverage(entries: list["FuturesEntry"], leverage: float) -> list["FuturesEntry"]:\n    """把所有部位改成同一個槓桿倍數，不再依停損距離決定。"""\n    return [FuturesEntry(e.entry_i, e.exit_i, leverage, e.entry_index,\n                         e.stop_distance, e.trade) for e in entries]\n\n\ndef hybrid_entries(entries: list["FuturesEntry"], bars, ma_period: int = 200,\n                   below_leverage: float = 3.0, above_risk_scale: float = 0.5,\n                   above_cap: float | None = None) -> list["FuturesEntry"]:\n    """§20 混合注碼：按進場日與均線的相對位置切換槓桿規則。\n\n        進場日收盤 < MA → 固定 `below_leverage`（深回檔是最好的機會，別壓小注）\n        其餘           → 風險式槓桿 × `above_risk_scale`（反環境訊號只給一半預算），\n                         可再以 `above_cap` 封頂\n\n    均線資料不足的日子歸「其餘」—— 沒有濾網資訊時維持風險式，是保守的選擇。\n    輸入的 `entries` 應來自 `entries_from_trades()`，其 `leverage` 即風險式槓桿。\n    """\n    from .engine import moving_average\n    ma = moving_average(bars, ma_period)\n    out = []\n    for e in entries:\n        m = ma[e.entry_i]\n        if m is not None and bars[e.entry_i].close < m:\n            lev = below_leverage\n        else:\n            lev = e.leverage * above_risk_scale\n            if above_cap is not None:\n                lev = min(lev, above_cap)\n        out.append(FuturesEntry(e.entry_i, e.exit_i, lev, e.entry_index,\n                                e.stop_distance, e.trade))\n    return out\n\n\ndef entry_regime(bars, i: int, ma: list) -> str:\n    """進場日 i 適用哪一種注碼：\'below\'（固定 3x）或 \'above\'（半預算風險式）。"""\n    m = ma[i]\n    return "below" if (m is not None and bars[i].close < m) else "above"\n\n\ndef ma_band_filter(index_close: list[float], ma: list[float | None],\n                   band_up: float = 0.0, band_dn: float = 0.0) -> list[bool]:\n    """核心部位的均線濾網，帶**遲滯緩衝帶**（hysteresis band）。\n\n        關 → 開：收盤 > 均線 × (1 + band_up)\n        開 → 關：收盤 ≤ 均線 × (1 − band_dn)\n\n    兩條門檻中間是無動作區，收盤在均線附近來回不會反覆換邊。這解決的是\n    「站上、跌破、再站上」的空轉，不是交易成本 —— 成本本來就只有十萬分之二\n    的期交稅加每口 50 元手續費（見 `FuturesCost`）。\n\n    `band_up = band_dn = 0` 時**完全等價**於舊版的 `收盤 > 均線`：\n    開啟條件用嚴格大於、關閉條件用小於等於，收盤恰好等於均線時兩者都判為關；\n    均線尚未成形（`ma[i] is None`）則一律判為關，不沿用前一天的狀態\n    —— 指標算不出來就不持有，也讓帶寬 0 逐點重現舊行為。\n\n    狀態只由「收盤與均線的相對位置」決定，與核心當下有沒有真的持有無關。\n    策略部位接手期間濾網照常更新，所以策略平倉後不必重新等一次向上突破。\n    """\n    out: list[bool] = []\n    on = False\n    for c, m in zip(index_close, ma):\n        if m is None:\n            on = False\n        elif not on and c > m * (1.0 + band_up):\n            on = True\n        elif on and c <= m * (1.0 - band_dn):\n            on = False\n        out.append(on)\n    return out\n\n\ndef core_overlay(dates: list[date], continuous: list[float],\n                 entries: list["FuturesEntry"], cost: FuturesCost,\n                 index_close: list[float], core_leverage: float,\n                 ma: list[float | None],\n                 step_gain: float | None = None,\n                 step_leverage: float = 1.0,\n                 band_up: float = 0.0,\n                 band_dn: float = 0.0) -> tuple[list[float], list[FuturesTrade], float]:\n    """在 `vehicle_series` 之上，空手期間補一個低槓桿的核心部位。\n\n    核心只在**策略沒有部位**且**均線濾網為開**時持有，策略一有訊號就先平掉核心。\n    逆勢濾網（`trend_filter(below=True)`）只在收盤**低於**均線時進場，\n    所以兩者天然互斥：低於均線做策略、高於均線抱核心，中間沒有重疊。\n\n    時序與策略一致：`ma`/收盤在第 i 天收盤後判定，部位在當天期貨收盤建立，\n    因此第 i 天的損益由**前一天**的判斷決定 —— 用當天判斷賺當天的報酬是前視偏誤。\n\n    Args:\n        band_up / band_dn: 均線濾網的遲滯緩衝帶，見 `ma_band_filter()`。\n            預設 0 等同舊版的 `收盤 > 均線`，逐日判定。\n\n    回傳 (淨值, 策略交易明細, 有部位的日子佔比)。核心部位不算成交易筆數。\n    """\n    on = ma_band_filter(index_close, ma, band_up, band_dn)\n    by_entry = {e.entry_i: e for e in entries}\n    nav = [1.0] * len(dates)\n    detail: list[FuturesTrade] = []\n    equity, active, eq_at_entry, anchor, f0, lev = 1.0, None, 0.0, 0.0, 0.0, 0.0\n    holding_core, exposed = False, 0\n    stepped, step_d = False, None\n\n    for i in range(len(dates)):\n        if active is not None:\n            v = anchor * (1.0 + lev * (continuous[i] / f0 - 1.0))\n            exposed += 1\n            closing = active.exit_i is not None and i == active.exit_i\n            if closing:\n                v *= 1.0 - lev * cost.one_way_rate(index_close[i])\n            elif (step_gain is not None and not stepped\n                    and v >= eq_at_entry * (1.0 + step_gain) and lev > step_leverage):\n                v = _step_down(v, lev, anchor, f0, continuous[i],\n                               cost.one_way_rate(index_close[i]), step_leverage)\n                anchor, f0, lev = v, continuous[i], step_leverage\n                stepped, step_d = True, dates[i]\n            nav[i] = v\n            if closing:\n                f_entry = continuous[active.entry_i]\n                detail.append(FuturesTrade(\n                    entry_date=dates[active.entry_i], exit_date=dates[i],\n                    entry_index=active.entry_index, entry_futures=f_entry,\n                    exit_futures=continuous[i], leverage=active.leverage,\n                    stop_distance=active.stop_distance,\n                    futures_return=continuous[i] / f_entry - 1.0,\n                    ret=v / eq_at_entry - 1.0,\n                    final_leverage=lev, step_date=step_d))\n                equity, active, holding_core = v, None, False\n            continue\n\n        if i > 0 and holding_core:                 # 昨收就持有核心 → 賺今天\n            equity *= 1.0 + core_leverage * (continuous[i] / continuous[i - 1] - 1.0)\n            exposed += 1\n\n        e = by_entry.get(i)\n        if e is not None:                          # 今收轉進策略部位\n            if holding_core:\n                equity *= 1.0 - core_leverage * cost.one_way_rate(index_close[i])\n                holding_core = False\n            active, eq_at_entry, lev = e, equity, e.leverage\n            anchor = equity * (1.0 - lev * cost.one_way_rate(index_close[i]))\n            f0 = continuous[i]\n            stepped, step_d = False, None\n            nav[i] = anchor\n            exposed += 1\n            continue\n\n        if on[i] != holding_core:                  # 核心進出，各付一次成本\n            equity *= 1.0 - core_leverage * cost.one_way_rate(index_close[i])\n            holding_core = on[i]\n        nav[i] = equity\n\n    if active is not None:\n        f_entry = continuous[active.entry_i]\n        detail.append(FuturesTrade(\n            entry_date=dates[active.entry_i], exit_date=None,\n            entry_index=active.entry_index, entry_futures=f_entry,\n            exit_futures=continuous[-1], leverage=active.leverage,\n            stop_distance=active.stop_distance,\n            futures_return=continuous[-1] / f_entry - 1.0,\n            ret=nav[-1] / eq_at_entry - 1.0,\n            final_leverage=lev, step_date=step_d))\n    return nav, detail, exposed / len(dates)\n'
SOURCES['__init__'] = '"""台灣指數「快速修復」回檔入場策略（標的：台灣50正2 / 00631L）。"""\n\nfrom .bars import Bar, load_csv\nfrom .config import DEFAULT_CONFIG, StrategyConfig\nfrom .engine import Engine, Result, Trade, position_size\nfrom .levels import Levels, build_levels\nfrom .plan import TradePlan, build_plan\nfrom .setup import FastRepairSetup, detect_setups\n\n__all__ = [\n    "Bar", "load_csv",\n    "StrategyConfig", "DEFAULT_CONFIG",\n    "Levels", "build_levels",\n    "FastRepairSetup", "detect_setups",\n    "Engine", "Result", "Trade", "position_size",\n    "TradePlan", "build_plan",\n]\n'

for _name, _src in SOURCES.items():
    (PKG / f'{_name}.py').write_text(_src, encoding='utf-8')

print(f'已寫入 {len(SOURCES)} 個模組到 tw_backdraw/')


## 4. 換倉價差資料

嵌入 332 筆歷史換倉日的「次月合約收盤價」（來源：期交所分月行情）。
遇到嵌入資料沒涵蓋的新換倉日（產生 notebook 之後才發生的），
下一節會自動向期交所補抓。


In [ ]:
import calendar, sys, urllib.parse, urllib.request

_ROLL_NEXT_CLOSE = {
"1999-01-19": 6433.0,
"1999-02-10": 5875.0,
"1999-03-16": 6730.0,
"1999-04-20": 7662.0,
"1999-05-18": 7643.0,
"1999-06-15": 7940.0,
"1999-07-20": 7825.0,
"1999-08-17": 8190.0,
"1999-09-14": 8130.0,
"1999-10-19": 7699.0,
"1999-11-16": 7655.0,
"1999-12-14": 7910.0,
"2000-01-18": 9353.0,
"2000-02-15": 10064.0,
"2000-03-14": 8925.0,
"2000-04-18": 9280.0,
"2000-05-16": 8805.0,
"2000-06-20": 8750.0,
"2000-07-18": 8360.0,
"2000-08-15": 7922.0,
"2000-09-19": 6890.0,
"2000-10-17": 5760.0,
"2000-11-14": 5870.0,
"2000-12-19": 5030.0,
"2001-01-16": 5765.0,
"2001-02-20": 5939.0,
"2001-03-20": 5655.0,
"2001-04-17": 5443.0,
"2001-05-15": 5170.0,
"2001-06-19": 5083.0,
"2001-07-17": 4356.0,
"2001-08-14": 4605.0,
"2001-09-14": 3705.0,
"2001-10-16": 3800.0,
"2001-11-20": 4437.0,
"2001-12-18": 5361.0,
"2002-01-15": 5549.0,
"2002-02-19": 5801.0,
"2002-03-19": 5874.0,
"2002-04-16": 6225.0,
"2002-05-14": 5763.0,
"2002-06-18": 5577.0,
"2002-07-16": 5270.0,
"2002-08-20": 4880.0,
"2002-09-17": 4643.0,
"2002-10-15": 4108.0,
"2002-11-19": 4740.0,
"2002-12-17": 4585.0,
"2003-01-14": 4993.0,
"2003-02-18": 4617.0,
"2003-03-18": 4545.0,
"2003-04-15": 4465.0,
"2003-05-20": 4196.0,
"2003-06-17": 4985.0,
"2003-07-15": 5313.0,
"2003-08-19": 5559.0,
"2003-09-16": 5716.0,
"2003-10-14": 5982.0,
"2003-11-18": 5982.0,
"2003-12-16": 5914.0,
"2004-01-16": 6322.0,
"2004-02-17": 6620.0,
"2004-03-16": 6570.0,
"2004-04-20": 6840.0,
"2004-05-18": 5450.0,
"2004-06-15": 5541.0,
"2004-07-20": 5233.0,
"2004-08-17": 5298.0,
"2004-09-14": 5929.0,
"2004-10-19": 5836.0,
"2004-11-16": 5931.0,
"2004-12-14": 5903.0,
"2005-01-18": 5957.0,
"2005-02-15": 6132.0,
"2005-03-15": 6065.0,
"2005-04-19": 5735.0,
"2005-05-17": 5855.0,
"2005-06-14": 6120.0,
"2005-07-19": 6390.0,
"2005-08-16": 6238.0,
"2005-09-20": 6120.0,
"2005-10-18": 5849.0,
"2005-11-15": 6033.0,
"2005-12-20": 6443.0,
"2006-01-17": 6694.0,
"2006-02-14": 6615.0,
"2006-03-14": 6419.0,
"2006-04-18": 6982.0,
"2006-05-16": 7005.0,
"2006-06-20": 6283.0,
"2006-07-18": 6150.0,
"2006-08-15": 6590.0,
"2006-09-19": 6867.0,
"2006-10-17": 7078.0,
"2006-11-14": 7198.0,
"2006-12-19": 7589.0,
"2007-01-16": 7833.0,
"2007-02-14": 7850.0,
"2007-03-20": 7708.0,
"2007-04-17": 7958.0,
"2007-05-15": 7938.0,
"2007-06-15": 8558.0,
"2007-07-17": 9458.0,
"2007-08-14": 8840.0,
"2007-09-17": 8850.0,
"2007-10-16": 9577.0,
"2007-11-20": 8720.0,
"2007-12-18": 7777.0,
"2008-01-15": 8501.0,
"2008-02-19": 7999.0,
"2008-03-18": 8043.0,
"2008-04-15": 8879.0,
"2008-05-20": 9073.0,
"2008-06-17": 8026.0,
"2008-07-15": 6650.0,
"2008-08-19": 6902.0,
"2008-09-16": 5657.0,
"2008-10-14": 5235.0,
"2008-11-18": 4130.0,
"2008-12-16": 4568.0,
"2009-01-20": 4171.0,
"2009-02-17": 4422.0,
"2009-03-17": 5027.0,
"2009-04-14": 5841.0,
"2009-05-19": 6673.0,
"2009-06-16": 6130.0,
"2009-07-14": 6488.0,
"2009-08-18": 6781.0,
"2009-09-15": 7338.0,
"2009-10-20": 7722.0,
"2009-11-17": 7698.0,
"2009-12-15": 7761.0,
"2010-01-19": 8230.0,
"2010-02-10": 7369.0,
"2010-03-16": 7656.0,
"2010-04-20": 7850.0,
"2010-05-18": 7578.0,
"2010-06-15": 7326.0,
"2010-07-20": 7586.0,
"2010-08-17": 7917.0,
"2010-09-14": 8103.0,
"2010-10-19": 7993.0,
"2010-11-16": 8273.0,
"2010-12-14": 8721.0,
"2011-01-18": 8952.0,
"2011-02-15": 8660.0,
"2011-03-15": 8232.0,
"2011-04-19": 8600.0,
"2011-05-17": 8896.0,
"2011-06-14": 8685.0,
"2011-07-19": 8418.0,
"2011-08-16": 7703.0,
"2011-09-20": 7468.0,
"2011-10-18": 7325.0,
"2011-11-15": 7505.0,
"2011-12-20": 6660.0,
"2012-01-17": 7212.0,
"2012-02-14": 7872.0,
"2012-03-20": 7986.0,
"2012-04-17": 7575.0,
"2012-05-15": 7369.0,
"2012-06-19": 7123.0,
"2012-07-17": 6983.0,
"2012-08-14": 7486.0,
"2012-09-18": 7750.0,
"2012-10-16": 7443.0,
"2012-11-20": 7125.0,
"2012-12-18": 7653.0,
"2013-01-15": 7721.0,
"2013-02-19": 7952.0,
"2013-03-19": 7815.0,
"2013-04-16": 7753.0,
"2013-05-14": 8278.0,
"2013-06-18": 7865.0,
"2013-07-16": 8134.0,
"2013-08-20": 7743.0,
"2013-09-17": 8220.0,
"2013-10-15": 8344.0,
"2013-11-19": 8232.0,
"2013-12-17": 8345.0,
"2014-01-14": 8535.0,
"2014-02-18": 8557.0,
"2014-03-18": 8718.0,
"2014-04-15": 8878.0,
"2014-05-20": 8883.0,
"2014-06-17": 9160.0,
"2014-07-15": 9465.0,
"2014-08-19": 9218.0,
"2014-09-16": 9162.0,
"2014-10-14": 8736.0,
"2014-11-18": 8883.0,
"2014-12-16": 8986.0,
"2015-01-20": 9275.0,
"2015-02-13": 9537.0,
"2015-03-17": 9568.0,
"2015-04-14": 9652.0,
"2015-05-19": 9705.0,
"2015-06-16": 9040.0,
"2015-07-14": 8848.0,
"2015-08-18": 8123.0,
"2015-09-15": 8193.0,
"2015-10-20": 8655.0,
"2015-11-17": 8435.0,
"2015-12-15": 8044.0,
"2016-01-19": 7837.0,
"2016-02-16": 8208.0,
"2016-03-15": 8569.0,
"2016-04-19": 8595.0,
"2016-05-17": 8118.0,
"2016-06-14": 8381.0,
"2016-07-19": 8921.0,
"2016-08-16": 9028.0,
"2016-09-20": 9110.0,
"2016-10-18": 9200.0,
"2016-11-15": 8917.0,
"2016-12-20": 9254.0,
"2017-01-17": 9333.0,
"2017-02-14": 9700.0,
"2017-03-14": 9742.0,
"2017-04-18": 9726.0,
"2017-05-16": 10009.0,
"2017-06-20": 10163.0,
"2017-07-18": 10344.0,
"2017-08-15": 10251.0,
"2017-09-19": 10554.0,
"2017-10-17": 10716.0,
"2017-11-14": 10673.0,
"2017-12-19": 10446.0,
"2018-01-16": 10979.0,
"2018-02-12": 10371.0,
"2018-03-20": 10986.0,
"2018-04-17": 10792.0,
"2018-05-15": 10852.0,
"2018-06-19": 10629.0,
"2018-07-17": 10660.0,
"2018-08-14": 10790.0,
"2018-09-18": 10779.0,
"2018-10-16": 9909.0,
"2018-11-20": 9711.0,
"2018-12-18": 9692.0,
"2019-01-15": 9794.0,
"2019-02-19": 10145.0,
"2019-03-19": 10495.0,
"2019-04-16": 10923.0,
"2019-05-14": 10525.0,
"2019-06-18": 10338.0,
"2019-07-16": 10727.0,
"2019-08-20": 10487.0,
"2019-09-17": 10839.0,
"2019-10-15": 11087.0,
"2019-11-19": 11640.0,
"2019-12-17": 12090.0,
"2020-01-14": 12155.0,
"2020-02-18": 11629.0,
"2020-03-17": 9190.0,
"2020-04-14": 10261.0,
"2020-05-19": 10809.0,
"2020-06-16": 11329.0,
"2020-07-14": 12033.0,
"2020-08-18": 12831.0,
"2020-09-15": 12779.0,
"2020-10-20": 12778.0,
"2020-11-17": 13581.0,
"2020-12-15": 13977.0,
"2021-01-19": 15847.0,
"2021-02-05": 15716.0,
"2021-03-16": 16240.0,
"2021-04-20": 17256.0,
"2021-05-18": 16082.0,
"2021-06-15": 17272.0,
"2021-07-20": 17341.0,
"2021-08-17": 16469.0,
"2021-09-14": 17401.0,
"2021-10-19": 16872.0,
"2021-11-16": 17680.0,
"2021-12-14": 17543.0,
"2022-01-18": 18314.0,
"2022-02-15": 17922.0,
"2022-03-15": 16825.0,
"2022-04-19": 16966.0,
"2022-05-17": 16025.0,
"2022-06-14": 15602.0,
"2022-07-19": 14528.0,
"2022-08-16": 15366.0,
"2022-09-20": 14528.0,
"2022-10-18": 13077.0,
"2022-11-15": 14465.0,
"2022-12-20": 14125.0,
"2023-01-17": 14910.0,
"2023-02-14": 15645.0,
"2023-03-14": 15286.0,
"2023-04-18": 15871.0,
"2023-05-16": 15632.0,
"2023-06-20": 17009.0,
"2023-07-18": 17172.0,
"2023-08-15": 16399.0,
"2023-09-19": 16625.0,
"2023-10-17": 16672.0,
"2023-11-14": 16968.0,
"2023-12-19": 17562.0,
"2024-01-16": 17361.0,
"2024-02-20": 18762.0,
"2024-03-19": 19884.0,
"2024-04-16": 19981.0,
"2024-05-14": 21014.0,
"2024-06-18": 22668.0,
"2024-07-16": 24000.0,
"2024-08-20": 22403.0,
"2024-09-16": 21828.0,
"2024-10-15": 23285.0,
"2024-11-19": 22894.0,
"2024-12-17": 23095.0,
"2025-01-14": 22816.0,
"2025-02-18": 23652.0,
"2025-03-18": 22259.0,
"2025-04-15": 19779.0,
"2025-05-20": 21451.0,
"2025-06-17": 21784.0,
"2025-07-15": 22675.0,
"2025-08-19": 24284.0,
"2025-09-16": 25639.0,
"2025-10-14": 26765.0,
"2025-11-18": 26805.0,
"2025-12-16": 27713.0,
"2026-01-20": 31800.0,
"2026-02-11": 33849.0,
"2026-03-17": 33910.0,
"2026-04-14": 36639.0,
"2026-05-19": 40383.0,
"2026-06-16": 45849.0,
"2026-07-14": 45135.0,
"2026-08-18": 45166.0
}

def fetch_taifex_next_close(day, contract):
    """向期交所補抓某日某合約的收盤價（嵌入資料沒涵蓋的新換倉日才會用到）。"""
    y, m = int(day[:4]), int(day[5:7])
    last = calendar.monthrange(y, m)[1]
    body = urllib.parse.urlencode({
        'down_type': '1', 'commodity_id': 'TX',
        'queryStartDate': f'{y:04d}/{m:02d}/01',
        'queryEndDate': f'{y:04d}/{m:02d}/{last:02d}'}).encode()
    try:
        req = urllib.request.Request(
            'https://www.taifex.com.tw/cht/3/futDataDown', data=body,
            headers={'User-Agent': 'Mozilla/5.0 (tw-backdraw)'})
        with urllib.request.urlopen(req, timeout=60) as resp:
            text = resp.read().decode('big5', errors='replace')
    except Exception as exc:
        print(f'  期交所補抓 {day} 失敗：{exc}', file=sys.stderr)
        return None
    for line in text.splitlines()[1:]:
        f = [x.strip() for x in line.split(',')]
        if (len(f) > 17 and f[1] == 'TX' and f[0].replace('/', '-') == day
                and f[2] == contract and f[17] == '一般' and f[6] not in ('-', '')):
            return float(f[6].replace(',', ''))
    return None

print(f'換倉價差：已嵌入 {len(_ROLL_NEXT_CLOSE)} 筆'
      f'（{min(_ROLL_NEXT_CLOSE)} ~ {max(_ROLL_NEXT_CLOSE)}）')


## 5. 取得資料

| 資料 | 來源 | 說明 |
|---|---|---|
| 加權指數 OHLC | `taiex_total_index:*` | 名稱易誤會，實際是**價格指數**，已與證交所核對相符 |
| 台指期近月收盤 | `futures_price:收盤價` 的 `TX一般` | 近月連續序列，1999 年起 |
| 到期月份 | `futures_price:到期月份(週別)` 的 `TX一般` | 用來偵測換倉日 |

換倉日以「近月收盤 ÷ 次月同日收盤」還原價差，接成可交易的連續序列。


In [ ]:
import datetime
import pandas as pd
from finlab import data
from tw_backdraw.bars import Bar
from tw_backdraw.futures import build_continuous, missing_rolls

idx = pd.DataFrame({k: data.get(f'taiex_total_index:{n}指數').iloc[:, 0]
                    for k, n in (('open', '開盤'), ('high', '最高'),
                                 ('low', '最低'), ('close', '收盤'))}
                   ).dropna(subset=['close'])
fc = data.get('futures_price:收盤價')['TX一般'].dropna()
fm = data.get('futures_price:到期月份(週別)')['TX一般'].dropna()
fdf = pd.DataFrame({'close': fc, 'exp': fm}).dropna()
fdf = fdf[fdf.index >= '1999-01-01']

fdates = [d.date() for d in fdf.index]
front = [float(x) for x in fdf['close']]
contract = [str(x) for x in fdf['exp']]

rolls = {datetime.date(*(int(x) for x in k.split('-'))): v
         for k, v in _ROLL_NEXT_CLOSE.items()}

# 嵌入資料沒涵蓋的換倉日（通常是產生 notebook 之後才發生的），向期交所補抓
for miss in missing_rolls(fdates, contract, rolls):
    i = fdates.index(miss)
    px = fetch_taifex_next_close(miss.isoformat(), contract[i + 1])
    if px:
        rolls[miss] = px
        print(f'  補抓換倉價差 {miss} → {contract[i + 1]} 收 {px:,.0f}')

still_missing = missing_rolls(fdates, contract, rolls)
cont = build_continuous(fdates, front, contract, rolls)
fut = dict(zip(fdates, cont))

bars = [Bar(d=d.date(), open=float(r['open']), high=float(r['high']),
            low=float(r['low']), close=float(r['close']))
        for d, r in idx.iterrows() if d.date() in fut]
fseries = [fut[b.d] for b in bars]

print(f'期間 {bars[0].d} ~ {bars[-1].d}（{len(bars)} 個交易日）')
if still_missing:
    print(f'⚠ 有 {len(still_missing)} 個換倉日缺次月報價，該處沿用原始跳動：'
          f'{still_missing[:5]}')
else:
    print('換倉價差：全數取得，無缺漏')


## 6. 選擇參數

| 變數 | 預設 | 說明 |
|---|---|---|
| `PRESET` | `tuned` | 參數組：tuned / post / balanced / winrate |
| `MAX_LEVERAGE` | 5.0 | 槓桿上限 |
| `SAME_DAY` | True | 當天期貨收盤成交；False 改隔一個交易日收盤（對照用） |

對應 repo 指令 `python3 scripts/futures_trades.py`（預設值完全相同）。


In [ ]:
from tw_backdraw.config import PRESETS

PRESET = 'tuned'       # ← 想換參數組改這裡
MAX_LEVERAGE = 5.0
SAME_DAY = True

cfg = PRESETS[PRESET]
print(f'訊號   回檔 ≥{cfg.setup.min_drawdown:.0%}，谷底起 ≤{cfg.setup.max_repair_bars} 日'
      f'內補回 ≥{cfg.setup.repair_fraction:.0%}')
print(f'槓桿   風險預算 {cfg.sizing.risk_per_trade:.0%} ÷ 停損距離，上限 {MAX_LEVERAGE:g}x')
print('成交   ' + ('當天期貨收盤（指數 13:30 收、期貨 13:45 收）' if SAME_DAY
               else '隔一個交易日收盤'))


## 7. 跑策略

訊號引擎跑在**加權指數**上，交易轉成**期貨部位**：
槓桿由停損距離決定、進出場各扣一次成本（期交稅十萬分之二＋每口 50 元），
持有期間權益線性於期貨報酬（固定口數，不複利）。


In [ ]:
from tw_backdraw.engine import Engine
from tw_backdraw.futures import (FuturesCost, entries_from_trades,
                                 trade_details, vehicle_series)

result = Engine(cfg).run(bars, fseries)
entries = entries_from_trades(result.trades, bars, cfg, MAX_LEVERAGE, SAME_DAY)
dates = [b.d for b in bars]
nav, detail = vehicle_series(dates, fseries, entries, FuturesCost(),
                             [b.close for b in bars])
details = trade_details(dates, nav, entries, detail)

print(f"{'進場':<12}{'出場':<12}{'停損距離':>9}{'槓桿':>7}"
      f"{'期貨報酬':>10}{'權益報酬':>10}")
print('-' * 62)
for t in detail:
    print(f"{t.entry_date!s:<12}{str(t.exit_date or '持有中'):<12}"
          f"{t.stop_distance:>9.2%}{t.leverage:>7.2f}"
          f"{t.futures_return:>10.1%}{t.ret:>10.1%}")

levs = [t.leverage for t in detail]
capped = sum(1 for x in levs if x >= MAX_LEVERAGE - 1e-9)
print(f'\n槓桿：中位數 {sorted(levs)[len(levs) // 2]:.2f}　'
      f'範圍 {min(levs):.2f}~{max(levs):.2f}　封頂 {capped}/{len(levs)} 筆')


## 8. 逐筆明細（含進場條件）

每筆列出：買賣日期、持有天數、進場當初的實際條件（前高／谷底／回檔幅度／
修復天數與比例／訊號日指數與距前高）、警戒線與失效線、距離停損 %、
據此算出的槓桿，以及權益報酬、最大報酬（MFE）、最大不利（MAE）、
期間最大回撤與出場原因。

`TRADES_SINCE = '2016-08-20'` 可只看這天之後的交易；`None` 列出全部。
對應 repo 指令 `python3 scripts/futures_trades.py --since 2016-08-20`。


In [ ]:
from tw_backdraw.futures import format_trade_details

TRADES_SINCE = None    # 例：'2016-08-20'

picked = details
if TRADES_SINCE:
    cut = datetime.datetime.strptime(TRADES_SINCE, '%Y-%m-%d').date()
    picked = [d for d in details if d.trade.entry_date >= cut]
    print(f'（只列出 {cut} 之後進場的交易）\n')
print(format_trade_details(picked))

rets = [d.trade.ret for d in picked]
wins = sum(1 for r in rets if r > 0)
dds = [d.max_drawdown for d in picked]
print(f'共 {len(rets)} 筆　勝 {wins} 敗 {len(rets) - wins}'
      f'（勝率 {wins / len(rets):.0%}）')
print(f'權益報酬：中位數 {sorted(rets)[len(rets) // 2]:+.1%}　'
      f'最佳 {max(rets):+.1%}　最差 {min(rets):+.1%}')
print(f'單筆期間最大回撤：中位數 {sorted(dds)[len(dds) // 2]:.1%}　'
      f'最深 {min(dds):.1%}')


## 9. FinLab 回測 → `report.display()`

把算好的期貨部位淨值交給 `sim()` 產生互動報表。兩個必要的繞法：

1. FinLab 內建的台股市場找不到 TXF，`report.display()` 組 positionConfig 會失敗
   —— 自訂一個市場類別，價格一律回傳我們的淨值序列。
2. FinLab 的 `position` 日期是**訊號日**、次一交易日成交，權重要往前挪一天。

> 成本已含在淨值裡（第 7 節），所以這裡 `fee_ratio=0`；
> `report.trade_at = 'close'` 是讓 `display()` 能序列化的顯示層標籤，不影響數值。


In [ ]:
import numpy as np
from finlab.backtest import sim
from finlab.markets.tw import TWMarket

SYMBOL = 'TXF'
di = pd.to_datetime([b.d for b in bars])
price = pd.DataFrame({SYMBOL: nav}, index=di)

class TWFuturesMarket(TWMarket):
    """自訂市場：價格一律回傳我們算好的期貨部位淨值。"""

    @staticmethod
    def get_name():
        return 'tw_futures'

    @staticmethod
    def get_asset_id_to_name():
        return {SYMBOL: '台指期連續合約'}

    def get_price(self, trade_at_price, adj=True):
        if isinstance(trade_at_price, (pd.DataFrame, pd.Series)):
            return pd.DataFrame(trade_at_price)
        return price

# FinLab 的 position 日期是「訊號日」、次一交易日成交，所以權重往前挪一天
pos = np.zeros(len(bars))
for e in entries:
    end = (e.exit_i - 1) if e.exit_i is not None else len(bars)
    pos[max(e.entry_i - 1, 0):max(end, 0)] = 1.0

report = sim(pd.DataFrame({SYMBOL: pos}, index=di), trade_at_price=price,
             position_limit=1, fee_ratio=0.0, tax_ratio=0.0,
             market=TWFuturesMarket(),
             name=f'台指期快速修復 {PRESET}（槓桿≤{MAX_LEVERAGE:g}x）',
             upload=False)
report.trade_at = 'close'          # 讓 display() 能序列化
report.display()


## 10. 與買進持有對照

**這張表是判斷策略有沒有價值的關鍵。** 策略與對照組使用同一套指標公式。
淨值用自己算的 `nav`，不用 `report.creturn` —— 後者從**第一筆交易**起算，
期初空手的策略 CAGR 會被灌水。


In [ ]:
def series_metrics(s):
    ret = s.pct_change().dropna()
    years = (s.index[-1] - s.index[0]).days / 365.25
    total = float(s.iloc[-1] / s.iloc[0] - 1)
    mdd = float((s / s.cummax() - 1).min())
    cagr = (1 + total) ** (1 / years) - 1
    down = ret[ret < 0].std()
    return dict(CAGR=cagr, 總報酬=total, 最大回檔=mdd,
                Sharpe=float(ret.mean() / ret.std() * np.sqrt(252)),
                Sortino=float(ret.mean() / down * np.sqrt(252)) if down else np.nan,
                Calmar=cagr / abs(mdd) if mdd else np.nan)

idx_series = pd.Series([b.close for b in bars], index=di)
fut_series = pd.Series(fseries, index=di)
eq = pd.Series(nav, index=di)
rows = {f'策略（槓桿≤{MAX_LEVERAGE:g}x）': series_metrics(eq),
        '買進持有 台指期（已還原換倉）': series_metrics(fut_series),
        '加權指數（價格指數）': series_metrics(idx_series)}

table = pd.DataFrame(rows).T
for c in ('CAGR', '總報酬', '最大回檔'):
    table[c] = table[c].map('{:.1%}'.format)
for c in ('Sharpe', 'Sortino', 'Calmar'):
    table[c] = table[c].map('{:.2f}'.format)
table


## 11. 停損假設 vs 實際

槓桿是按「距離停損 X% × L = 風險預算」定的，但實際虧損常大於假設：
訊號在收盤才判定（收盤已跌過線），且期貨與指數的 15 分鐘差內價格會續走。
這就是 5 倍上限仍出現單筆 −22% 的原因。


In [ ]:
worst = min(detail, key=lambda t: t.ret)
print(f'單筆最差：{worst.entry_date} → {worst.exit_date}　'
      f'槓桿 {worst.leverage:.2f}x　權益 {worst.ret:.1%}\n')
i_of = {b.d: i for i, b in enumerate(bars)}
for t in sorted(detail, key=lambda t: t.ret)[:5]:
    a = i_of[t.entry_date]
    b = i_of[t.exit_date] if t.exit_date else len(bars) - 1
    real = (bars[b].close - bars[a].close) / bars[a].close
    ratio = abs(real) / t.stop_distance if t.stop_distance else 0
    print(f'  {t.entry_date}  假設 -{t.stop_distance:.2%} → 實際 {real:+.2%}'
          f'（{ratio:.1f} 倍）　槓桿 {t.leverage:.1f}x → 權益 {t.ret:.1%}')


## 12.（選配）逆勢濾網與固定槓桿

對應 dist 腳本的 `--defensive` 與 `--fixed-leverage`：
只做進場日收盤**低於** MA200 的訊號（逆勢策略，深回檔才是報酬最好的場景），
並把所有部位改成固定 3 倍。這是 docs/strategy.md §18 討論的變體，
不影響上面主結果。


In [ ]:
from tw_backdraw.futures import fixed_leverage, trend_filter

ent_d = trend_filter(entries, bars, 200, below=True)
print(f'逆勢濾網：只做收盤低於 MA200 的訊號　{len(entries)} → {len(ent_d)} 筆')
ent_d = fixed_leverage(ent_d, 3.0)
nav_d, detail_d = vehicle_series(dates, fseries, ent_d, FuturesCost(),
                                 [b.close for b in bars])
rows_d = {'現行（停損距離定槓桿≤5x）': series_metrics(eq),
          '逆勢 MA200 ＋固定 3x': series_metrics(pd.Series(nav_d, index=di))}
table_d = pd.DataFrame(rows_d).T
for c in ('CAGR', '總報酬', '最大回檔'):
    table_d[c] = table_d[c].map('{:.1%}'.format)
for c in ('Sharpe', 'Sortino', 'Calmar'):
    table_d[c] = table_d[c].map('{:.2f}'.format)
table_d


---
## 已知限制

1. **樣本數少。** 1999 年起約 30 筆訊號，預設參數屬樣本內結果。
2. **停損假設會被跳空與收盤判定放大**，見第 11 節 —— 5 倍上限
   對應的實際單筆最差是 −22%，不是風險預算的 8%。
3. **換倉價差還原依賴期交所次月報價**；缺資料的換倉日退回原始跳動
   （第 5 節會列出是哪幾天，目前無缺漏）。
4. **保證金與強制平倉未建模。** 權益按線性攤算，實務上深度虧損時
   會先收到追繳通知。

完整討論見 repo 的 `docs/strategy.md` 與 `docs/tx_evaluation.md`。
